
# 01 — User Population Audit

## 1. Introduction

**Objective:** Conduct a comprehensive behavioral analysis of CERT r6.2 users before designing new attack simulation scenarios.

This notebook focuses on:

- **Population Overview:** Identify users with valid baselines and available future workdays.
- **Historical Channel Usage:** Analyze Email, USB, and HTTP activity during the baseline period.
- **Future Channel Activity:** Examine channel usage and availability during the future period.
- **Multi-Channel Behavior:** Investigate simultaneous and separate-day channel usage.
- **Activity Frequency and Intensity:** Analyze activity volume, frequency, and variability.
- **Temporal Patterns:** Examine activity continuity, inactivity gaps, and natural channel transitions.
- **Behavioral Segmentation:** Identify different user behavior profiles.

### Analysis Principles

- Only users with valid historical baselines will be considered.
- Baseline and future activity will be analyzed separately.
- Missing channel activity does not automatically make a user ineligible.
- No attack scenario constraints will be imposed at this stage.
- Original Data Engineering datasets will remain unchanged.

### Expected Outputs

- `user_population_profile.parquet`
- `population_audit_summary.md`

All final outputs will be saved under:

`reports/01_User_Population_Audit/`


In [1]:

# ============================================================
# 1. PROJECT SETUP AND DATA SOURCES
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

NOTEBOOK_NAME = "01_User_Population_Audit"

DATA_ROOT = (
    PROJECT_ROOT.parent
    / "Data_Engineering"
    / "Data_engineering"
    / "data"
)

REPORT_DIR = PROJECT_ROOT / "reports" / NOTEBOOK_NAME

DATASETS = {
    "user_day": DATA_ROOT / "processed" / "user_day_r62_v1.parquet",
    "baseline": DATA_ROOT / "features" / "user_baseline_r62_v1.parquet",
    "features": DATA_ROOT / "features" / "behavior_features_r62_v1.parquet",
    "host_eligibility": DATA_ROOT / "features" / "attack_host_eligibility_r62.parquet"
}

# Verify source paths
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(PROJECT_ROOT)

for name, path in DATASETS.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

# Create dedicated output directory
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("STATUS: PASS — Project setup completed.")
print("Source datasets verified: 4/4")
print("Report directory:", REPORT_DIR)


STATUS: PASS — Project setup completed.
Source datasets verified: 4/4
Report directory: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\01_User_Population_Audit



## 2. Population Overview

### 2.1 Baseline and Future Workday Coverage

**Objective:** Understand the available user population based on historical baselines and future workday availability.

This analysis will:

- Identify users with valid historical baselines.
- Count users with at least one eligible future workday.
- Identify users with no future workdays.
- Examine the distribution of baseline and future workdays across users.
- Establish the population for subsequent behavioral analysis.

**Important:**
- A valid user baseline is required.
- Missing channel-specific baseline statistics do not automatically invalidate a user.
- No attack duration, channel, or activity thresholds will be applied.
- All users will remain represented in the population analysis.

## 2. Population Overview

### 2.1 Baseline and Future Workday Coverage

**Objective:** Understand the available user population based on historical baselines and future workday availability.

This analysis will:

- Identify users with valid historical baselines.
- Count users with at least one eligible future workday.
- Identify users with no future workdays.
- Examine the distribution of baseline and future workdays across users.
- Establish the population for subsequent behavioral analysis.

**Important:**
- A valid user baseline is required.
- Missing channel-specific baseline statistics do not automatically invalidate a user.
- No attack duration, channel, or activity thresholds will be applied.
- All users will remain represented in the population analysis.


In [2]:

# ============================================================
# 2.1 BASELINE AND FUTURE WORKDAY COVERAGE
# ============================================================

import pandas as pd

# Load baseline population
baseline = pd.read_parquet(
    DATASETS["baseline"],
    columns=[
        "user",
        "baseline_period_start",
        "baseline_period_end",
        "baseline_workdays"
    ]
)

# Load future user-day keys
future = pd.read_parquet(
    DATASETS["features"],
    columns=["user", "day"]
)

# Count future workdays per user
future_counts = (
    future.groupby("user")
    .size()
    .rename("future_workdays")
)

# Build population profile
population_profile = baseline.merge(
    future_counts,
    on="user",
    how="left",
    validate="one_to_one"
)

population_profile["future_workdays"] = (
    population_profile["future_workdays"]
    .fillna(0)
    .astype(int)
)

# Identify valid historical baselines
population_profile["has_valid_baseline"] = (
    (population_profile["baseline_workdays"] > 0)
    & (
        pd.to_datetime(
            population_profile["baseline_period_start"]
        ) == pd.Timestamp("2010-01-02")
    )
    & (
        pd.to_datetime(
            population_profile["baseline_period_end"]
        ) == pd.Timestamp("2010-12-31")
    )
)

population_profile["has_future_workdays"] = (
    population_profile["future_workdays"] > 0
)

# Population summary
valid_baseline = population_profile["has_valid_baseline"]
future_available = population_profile["has_future_workdays"]

print("=" * 65)
print("POPULATION OVERVIEW")
print("=" * 65)

print(f"Baseline Users        : {len(population_profile):,}")
print(f"Valid Baseline Users  : {valid_baseline.sum():,}")
print(f"Users with Future Days: {(valid_baseline & future_available).sum():,}")
print(f"Users without Future  : {(valid_baseline & ~future_available).sum():,}")

# Descriptive workday distribution
print("\nWORKDAY DISTRIBUTION")
print("-" * 65)

active_population = population_profile.loc[
    valid_baseline & future_available
]

distribution = active_population[
    ["baseline_workdays", "future_workdays"]
].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90]
)

print(distribution.round(2).to_string())

print("\nSTATUS: Population overview completed.")


POPULATION OVERVIEW
Baseline Users        : 3,995
Valid Baseline Users  : 3,995
Users with Future Days: 3,737
Users without Future  : 258

WORKDAY DISTRIBUTION
-----------------------------------------------------------------
       baseline_workdays  future_workdays
count            3737.00          3737.00
mean              251.40           103.60
std                 1.49            10.02
min               249.00             1.00
25%               251.00           105.00
50%               251.00           105.00
75%               251.00           105.00
90%               251.00           105.00
max               260.00           108.00

STATUS: Population overview completed.



### 2.2 Future Workday Availability Distribution

**Objective:** Examine differences in future workday availability across the benign user population.

This analysis will:
- Group users by the number of available future workdays.
- Identify users with limited temporal availability.
- Examine the most common workday counts.
- Preserve all users without applying attack eligibility thresholds.

**Important:** Workday availability does not indicate activity in Email, USB, or HTTP channels.


In [3]:

# ============================================================
# 2.2 FUTURE WORKDAY AVAILABILITY DISTRIBUTION
# ============================================================

workdays = population_profile["future_workdays"]

# Descriptive groups (not eligibility thresholds)
groups = {
    "0 days": workdays.eq(0),
    "1–30 days": workdays.between(1, 30),
    "31–60 days": workdays.between(31, 60),
    "61–90 days": workdays.between(61, 90),
    "91–104 days": workdays.between(91, 104),
    "105 days": workdays.eq(105),
    "106+ days": workdays.ge(106)
}

distribution = pd.DataFrame([
    {
        "group": name,
        "users": int(mask.sum()),
        "percentage": round(
            mask.sum() / len(workdays) * 100, 2
        )
    }
    for name, mask in groups.items()
])

print("FUTURE WORKDAY AVAILABILITY")
print("=" * 55)
print(distribution.to_string(index=False))

print("\nMOST COMMON WORKDAY COUNTS")
print("-" * 55)

print(
    workdays.value_counts()
    .head(10)
    .to_string()
)

print("\nTotal users:", len(workdays))


FUTURE WORKDAY AVAILABILITY
      group  users  percentage
     0 days    258        6.46
  1–30 days     31        0.78
 31–60 days     30        0.75
 61–90 days     31        0.78
91–104 days     35        0.88
   105 days   3354       83.95
  106+ days    256        6.41

MOST COMMON WORKDAY COUNTS
-------------------------------------------------------
future_workdays
105    3354
0       258
106     168
107      86
104      26
50        5
30        4
20        4
37        3
61        3

Total users: 3995



## 3. Future Channel Activity Analysis

### 3.1 Channel Usage Distribution

**Objective:** Analyze the actual usage of Email, USB, and HTTP Upload during the Future Attack Period.

This analysis will:

- Count users who performed activity in each channel.
- Measure active days per channel for each user.
- Identify users with activity in one, two, or three channels.
- Identify users with no activity in the three target channels.
- Calculate channel activity rates relative to available future workdays.

**Channel Definitions:**
- Email: `external_email_count > 0`
- USB: `usb_copy_count > 0`
- HTTP: `www_upload_count > 0`

**Important:**
- Only the Future Period is analyzed.
- The original Baseline remains unchanged.
- Channel inactivity does not automatically disqualify users.
- No attack eligibility rules are applied at this stage.

The results will be retained in memory for subsequent behavioral analyses.


In [4]:

# ============================================================
# 3.1 FUTURE CHANNEL USAGE DISTRIBUTION
# ============================================================

import pandas as pd

# Load only relevant Future activity measurements
future_activity = pd.read_parquet(
    DATASETS["features"],
    columns=[
        "user",
        "day",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count"
    ]
)

# Identify natural activity in each channel
CHANNELS = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

for channel, column in CHANNELS.items():
    future_activity[f"{channel}_active"] = (
        future_activity[column] > 0
    ).astype("int8")

# Count active days per channel and user
activity_columns = [
    f"{channel}_active"
    for channel in CHANNELS
]

channel_days = (
    future_activity
    .groupby("user")[activity_columns]
    .sum()
    .rename(columns={
        f"{channel}_active": f"{channel}_days"
        for channel in CHANNELS
    })
    .reset_index()
)

# Update the existing population profile
new_columns = [f"{channel}_days" for channel in CHANNELS]

population_profile = (
    population_profile
    .drop(columns=new_columns + ["active_channel_count"],
          errors="ignore")
    .merge(
        channel_days,
        on="user",
        how="left",
        validate="one_to_one"
    )
)

population_profile[new_columns] = (
    population_profile[new_columns]
    .fillna(0)
    .astype(int)
)

# Number of distinct channels used during Future
population_profile["active_channel_count"] = (
    population_profile[new_columns].gt(0).sum(axis=1)
)

# Analyze users with eligible Future workdays
active_users = population_profile[
    population_profile["future_workdays"] > 0
]

# ------------------------------------------------------------
# SUMMARY 1: CHANNEL USAGE
# ------------------------------------------------------------

channel_summary = []

for channel in CHANNELS:

    days_col = f"{channel}_days"

    channel_summary.append({
        "channel": channel.upper(),
        "users_with_activity": int(
            (active_users[days_col] > 0).sum()
        ),
        "total_active_days": int(
            active_users[days_col].sum()
        ),
        "median_active_days": float(
            active_users[days_col].median()
        ),
        "users_pct": round(
            (active_users[days_col] > 0).mean() * 100, 2
        )
    })

channel_summary = pd.DataFrame(channel_summary)

# ------------------------------------------------------------
# SUMMARY 2: NUMBER OF CHANNELS USED PER USER
# ------------------------------------------------------------

channel_distribution = (
    active_users["active_channel_count"]
    .value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .rename_axis("channels_used")
    .reset_index(name="users")
)

print("=" * 65)
print("FUTURE CHANNEL ACTIVITY ANALYSIS")
print("=" * 65)

print(f"\nUsers Analyzed: {len(active_users):,}")

print("\nCHANNEL USAGE")
print("-" * 65)
print(channel_summary.to_string(index=False))

print("\nNUMBER OF CHANNELS USED PER USER")
print("-" * 65)
print(channel_distribution.to_string(index=False))

print("\nSTATUS: Channel usage analysis completed.")


FUTURE CHANNEL ACTIVITY ANALYSIS

Users Analyzed: 3,737

CHANNEL USAGE
-----------------------------------------------------------------
channel  users_with_activity  total_active_days  median_active_days  users_pct
  EMAIL                 3737             221839                70.0     100.00
    USB                  719              30018                 0.0      19.24
   HTTP                  567              13898                 0.0      15.17

NUMBER OF CHANNELS USED PER USER
-----------------------------------------------------------------
 channels_used  users
             0      0
             1   2742
             2    704
             3    291

STATUS: Channel usage analysis completed.



### 3.2 Same-Day Multi-Channel Activity

**Objective:** Examine how often users naturally perform activities across multiple channels on the same eligible workday during the Future period.

This analysis will:

- Measure the number of active channels per user-day.
- Identify days with zero, one, two, or three active channels.
- Count users who have at least one day with multi-channel activity.
- Examine the frequency of two-channel and three-channel activity for each user.
- Distinguish historical channel diversity from same-day channel overlap.

**Important:** Same-day multi-channel activity is a behavioral characteristic, not an attack eligibility requirement.

No attack rules will be applied, and no output files will be generated at this stage.


In [5]:

# ============================================================
# 3.2 SAME-DAY MULTI-CHANNEL ACTIVITY
# ============================================================

# Count active channels on each eligible user-day
future_activity["daily_active_channels"] = (
    future_activity[
        ["email_active", "usb_active", "http_active"]
    ].sum(axis=1)
)

# Daily overlap indicators
future_activity["two_plus_channels"] = (
    future_activity["daily_active_channels"] >= 2
).astype("int8")

future_activity["three_channels"] = (
    future_activity["daily_active_channels"] == 3
).astype("int8")

future_activity["any_channel"] = (
    future_activity["daily_active_channels"] >= 1
).astype("int8")

# Summarize per-user behavior
overlap_profile = (
    future_activity.groupby("user")
    .agg(
        any_channel_days=("any_channel", "sum"),
        multi_channel_days=("two_plus_channels", "sum"),
        three_channel_days=("three_channels", "sum")
    )
    .reset_index()
)

# Update population profile
overlap_cols = [
    "any_channel_days",
    "multi_channel_days",
    "three_channel_days"
]

population_profile = (
    population_profile
    .drop(columns=overlap_cols, errors="ignore")
    .merge(
        overlap_profile,
        on="user",
        how="left",
        validate="one_to_one"
    )
)

population_profile[overlap_cols] = (
    population_profile[overlap_cols]
    .fillna(0)
    .astype(int)
)

# Daily distribution
daily_distribution = (
    future_activity["daily_active_channels"]
    .value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .rename_axis("active_channels")
    .reset_index(name="workdays")
)

# Users with same-day overlap
active_users = population_profile[
    population_profile["future_workdays"] > 0
]

print("=" * 65)
print("SAME-DAY MULTI-CHANNEL ACTIVITY")
print("=" * 65)

print("\nDAILY CHANNEL DISTRIBUTION")
print("-" * 65)
print(daily_distribution.to_string(index=False))

print("\nUSER-LEVEL OVERLAP")
print("-" * 65)

print(
    "Users with at least one 2+ channel day:",
    f"{(active_users['multi_channel_days'] > 0).sum():,}"
)

print(
    "Users with at least one 3-channel day:",
    f"{(active_users['three_channel_days'] > 0).sum():,}"
)

print(
    "Users with no multi-channel days:",
    f"{(active_users['multi_channel_days'] == 0).sum():,}"
)

print("\nMULTI-CHANNEL DAYS PER USER")
print("-" * 65)

print(
    active_users[
        ["multi_channel_days", "three_channel_days"]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    ).round(2).to_string()
)


SAME-DAY MULTI-CHANNEL ACTIVITY

DAILY CHANNEL DISTRIBUTION
-----------------------------------------------------------------
 active_channels  workdays
               0    147442
               1    215095
               2     23215
               3      1410

USER-LEVEL OVERLAP
-----------------------------------------------------------------
Users with at least one 2+ channel day: 985
Users with at least one 3-channel day: 239
Users with no multi-channel days: 2,752

MULTI-CHANNEL DAYS PER USER
-----------------------------------------------------------------
       multi_channel_days  three_channel_days
count             3737.00             3737.00
mean                 6.59                0.38
std                 15.12                2.07
min                  0.00                0.00
25%                  0.00                0.00
50%                  0.00                0.00
75%                  3.00                0.00
90%                 24.00                0.00
95%              


### 3.3 Cross-Day Channel Diversity and Activity Duration

**Objective:** Analyze user channel diversity and quantify the number of activity days associated with each behavioral pattern during the Future period.

**This analysis will:**
- Classify users based on single-channel, same-day two-channel, same-day three-channel, and separate-day multi-channel activity.
- Count the users belonging to each behavioral group.
- Calculate total days with zero, one, two, or three active channels within each group.
- Examine the distribution of relevant activity days per user, including mean, median, percentiles, and maximum.
- Distinguish channel diversity across different days from simultaneous channel activity.

**Important:**
- All measurements are based on natural Future-period activity.
- Each day represents one eligible user-workday.
- Behavioral groups are descriptive, not attack eligibility categories.
- No simulation rules or thresholds are applied.


In [8]:

# ============================================================
# 3.3 CROSS-DAY CHANNEL DIVERSITY AND ACTIVITY DURATION
# ============================================================

# ------------------------------------------------------------
# 1. COUNT DAYS BY NUMBER OF ACTIVE CHANNELS
# ------------------------------------------------------------

daily_channel_counts = (
    future_activity
    .groupby("user")["daily_active_channels"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1, 2, 3], fill_value=0)
)

daily_channel_counts.columns = [
    "days_0ch",
    "days_1ch",
    "days_2ch",
    "days_3ch"
]

# Update the population profile
for col in daily_channel_counts.columns:
    population_profile[col] = (
        population_profile["user"]
        .map(daily_channel_counts[col])
        .fillna(0)
        .astype(int)
    )

# ------------------------------------------------------------
# 2. CLASSIFY USER BEHAVIOR
# ------------------------------------------------------------

population_profile["channel_overlap_pattern"] = (
    "No target-channel activity"
)

population_profile.loc[
    population_profile["active_channel_count"] == 1,
    "channel_overlap_pattern"
] = "Single channel"

population_profile.loc[
    (population_profile["active_channel_count"] >= 2)
    & (population_profile["days_2ch"] == 0)
    & (population_profile["days_3ch"] == 0),
    "channel_overlap_pattern"
] = "Multiple channels on separate days only"

population_profile.loc[
    (population_profile["days_2ch"] > 0)
    & (population_profile["days_3ch"] == 0),
    "channel_overlap_pattern"
] = "Same-day two-channel overlap"

population_profile.loc[
    population_profile["days_3ch"] > 0,
    "channel_overlap_pattern"
] = "Same-day three-channel overlap"

# Only users with Future workdays
active_users = population_profile.loc[
    population_profile["future_workdays"] > 0
].copy()

# ------------------------------------------------------------
# 3. GROUP SUMMARY: USERS AND TOTAL DAYS
# ------------------------------------------------------------

group_summary = (
    active_users
    .groupby("channel_overlap_pattern")
    .agg(
        users=("user", "size"),
        days_0ch=("days_0ch", "sum"),
        days_1ch=("days_1ch", "sum"),
        days_2ch=("days_2ch", "sum"),
        days_3ch=("days_3ch", "sum")
    )
    .reset_index()
)

group_summary["users_pct"] = (
    group_summary["users"] / len(active_users) * 100
).round(2)

print("=" * 75)
print("CROSS-DAY CHANNEL DIVERSITY")
print("=" * 75)

print("\nUSER GROUPS AND TOTAL ACTIVITY DAYS")
print("-" * 75)
print(group_summary.to_string(index=False))

# ------------------------------------------------------------
# 4. PER-USER ACTIVITY DAY DISTRIBUTIONS
# ------------------------------------------------------------

# Relevant day type for each behavioral group
GROUP_METRICS = {
    "Single channel": (
        "days_1ch",
        "Days with exactly 1 active channel"
    ),
    "Same-day two-channel overlap": (
        "days_2ch",
        "Days with exactly 2 active channels"
    ),
    "Same-day three-channel overlap": (
        "days_3ch",
        "Days with exactly 3 active channels"
    ),
    "Multiple channels on separate days only": (
        "days_1ch",
        "Days with 1 active channel (across different days)"
    )
}

distribution_records = []

for group_name, (column, description) in GROUP_METRICS.items():

    users = active_users.loc[
        active_users["channel_overlap_pattern"] == group_name
    ]

    if users.empty:
        continue

    values = users[column]

    distribution_records.append({
        "group": group_name,
        "users": len(users),
        "day_type": description,
        "total_days": int(values.sum()),
        "min": int(values.min()),
        "p25": values.quantile(0.25),
        "median": values.median(),
        "mean": values.mean(),
        "p75": values.quantile(0.75),
        "p90": values.quantile(0.90),
        "max": int(values.max())
    })

duration_summary = pd.DataFrame(distribution_records)

print("\nACTIVITY DAY DISTRIBUTION PER USER GROUP")
print("-" * 75)

print(
    duration_summary[
        [
            "group", "users", "total_days",
            "min", "p25", "median",
            "mean", "p75", "p90", "max"
        ]
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

day_columns = [
    "days_0ch", "days_1ch",
    "days_2ch", "days_3ch"
]

assert (
    active_users[day_columns].sum(axis=1)
    == active_users["future_workdays"]
).all(), "Workday totals do not match."

assert (
    group_summary["users"].sum() == len(active_users)
), "Population classification mismatch."

print("\nVALIDATION: PASS")
print("Total classified users:", len(active_users))
print("Total user-workdays:", int(active_users["future_workdays"].sum()))


CROSS-DAY CHANNEL DIVERSITY

USER GROUPS AND TOTAL ACTIVITY DAYS
---------------------------------------------------------------------------
                channel_overlap_pattern  users  days_0ch  days_1ch  days_2ch  days_3ch  users_pct
Multiple channels on separate days only     10       819       232         0         0       0.27
         Same-day three-channel overlap    239      4868     11814      6982      1410       6.40
           Same-day two-channel overlap    746     21117     39920     16233         0      19.96
                         Single channel   2742    120638    163129         0         0      73.37

ACTIVITY DAY DISTRIBUTION PER USER GROUP
---------------------------------------------------------------------------
                                  group  users  total_days  min   p25  median  mean  p75  p90  max
                         Single channel   2742      163129    1 40.00    70.0 59.49 79.0 85.0   99
           Same-day two-channel overlap    746       


### 3.4 Channel Activity Frequency

**Objective:** Analyze how frequently users perform natural activity in Email, USB, and HTTP Upload during the Future Attack Period.

**This analysis will:**
- Measure the number of active days per channel.
- Calculate activity frequency relative to available future workdays.
- Examine the distribution of active days among channel users.
- Identify differences between frequent and occasional channel usage.

**Important:**
- Only the Future period is analyzed.
- Users without activity in a particular channel are retained.
- No attack eligibility thresholds are applied.
- No output files are generated at this stage.


In [9]:

# ============================================================
# 3.4 CHANNEL ACTIVITY FREQUENCY
# ============================================================

import pandas as pd

# Users with eligible Future workdays
active_users = population_profile.loc[
    population_profile["future_workdays"] > 0
].copy()

CHANNELS = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

# Calculate activity rates for each channel
for channel in CHANNELS:
    rate_col = f"{channel}_activity_rate"

    population_profile[rate_col] = (
        population_profile[f"{channel}_days"]
        / population_profile["future_workdays"].replace(0, float("nan"))
    )

    active_users[rate_col] = (
        active_users[f"{channel}_days"]
        / active_users["future_workdays"]
    )

# Summarize frequency among users active in each channel
frequency_records = []

for channel in CHANNELS:

    days_col = f"{channel}_days"
    rate_col = f"{channel}_activity_rate"

    channel_users = active_users.loc[
        active_users[days_col] > 0
    ]

    frequency_records.append({
        "channel": channel.upper(),
        "users": len(channel_users),
        "min_days": channel_users[days_col].min(),
        "p25_days": channel_users[days_col].quantile(0.25),
        "median_days": channel_users[days_col].median(),
        "mean_days": channel_users[days_col].mean(),
        "p75_days": channel_users[days_col].quantile(0.75),
        "p90_days": channel_users[days_col].quantile(0.90),
        "max_days": channel_users[days_col].max(),
        "median_rate_pct": channel_users[rate_col].median() * 100
    })

frequency_summary = pd.DataFrame(frequency_records)

print("=" * 70)
print("CHANNEL ACTIVITY FREQUENCY — FUTURE PERIOD")
print("=" * 70)

print("\nFREQUENCY AMONG ACTIVE CHANNEL USERS")
print("-" * 70)

print(
    frequency_summary.round(2).to_string(index=False)
)

print("\nUSERS WITHOUT ACTIVITY IN EACH CHANNEL")
print("-" * 70)

for channel in CHANNELS:
    inactive = (
        active_users[f"{channel}_days"] == 0
    ).sum()

    print(f"{channel.upper():<8}: {inactive:,}")

print("\nSTATUS: Frequency analysis completed.")


CHANNEL ACTIVITY FREQUENCY — FUTURE PERIOD

FREQUENCY AMONG ACTIVE CHANNEL USERS
----------------------------------------------------------------------
channel  users  min_days  p25_days  median_days  mean_days  p75_days  p90_days  max_days  median_rate_pct
  EMAIL   3737         1      39.0         70.0      59.36      79.0      85.0       104            67.62
    USB    719         1      17.0         33.0      41.75      64.0      91.2       107            31.43
   HTTP    567         1      20.0         24.0      24.51      29.0      32.0        40            23.81

USERS WITHOUT ACTIVITY IN EACH CHANNEL
----------------------------------------------------------------------
EMAIL   : 0
USB     : 3,018
HTTP    : 3,170

STATUS: Frequency analysis completed.



### 3.5 Channel Activity Intensity

**Objective:** Analyze the volume and variability of natural activity in each channel during active Future workdays.

**This analysis will:**
- Calculate total operations per user and channel.
- Measure mean, median, P95, and maximum daily operations.
- Examine differences in activity intensity across users.
- Analyze external email bytes separately from event counts.
- Identify users with low-volume versus high-volume activity.

**Channel Measurements:**
- Email: External email count and external email bytes.
- USB: USB copy count.
- HTTP: WWW upload count.

**Important:**
- Statistics are calculated using active channel days only.
- Users with no channel activity remain in the population profile.
- HTTP and USB operation counts do not represent transferred bytes.
- No attack thresholds or simulation rules are applied.
- Only Future data is analyzed. The Baseline remains unchanged.


In [11]:

# ============================================================
# 3.5 CHANNEL ACTIVITY INTENSITY
# ============================================================

import pandas as pd

# Read Future activity only
activity = pd.read_parquet(
    DATASETS["features"],
    columns=[
        "user",
        "external_email_count",
        "external_email_bytes",
        "usb_copy_count",
        "www_upload_count"
    ]
)

CHANNEL_METRICS = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

intensity_records = []

# ------------------------------------------------------------
# 1. PER-USER CHANNEL INTENSITY
# ------------------------------------------------------------

for channel, metric in CHANNEL_METRICS.items():

    # Only days with activity in this channel
    active = activity.loc[
        activity[metric] > 0
    ]

    grouped = active.groupby("user")[metric]

    stats = grouped.agg(
        active_days="size",
        total="sum",
        mean="mean",
        median="median",
        maximum="max"
    )

    stats["p95"] = grouped.quantile(0.95)

    # Verify consistency with previous frequency analysis
    expected_days = population_profile["user"].map(
        stats["active_days"]
    ).fillna(0).astype(int)

    assert (
        expected_days.to_numpy()
        == population_profile[f"{channel}_days"].to_numpy()
    ).all(), f"{channel} active-day mismatch"

    # Add measurements to the main user profile
    for stat in ["total", "mean", "median", "p95", "maximum"]:

        col_name = f"{channel}_{stat}_intensity"

        population_profile[col_name] = (
            population_profile["user"].map(stats[stat])
        )

    # Zero activity means zero total operations
    total_col = f"{channel}_total_intensity"

    population_profile[total_col] = (
        population_profile[total_col].fillna(0)
    )

    # Summary among actual channel users
    user_means = stats["mean"]

    intensity_records.append({
        "channel": channel.upper(),
        "users": len(stats),
        "median_total_ops": stats["total"].median(),
        "median_daily_mean": user_means.median(),
        "p25_daily_mean": user_means.quantile(0.25),
        "p75_daily_mean": user_means.quantile(0.75),
        "p90_daily_mean": user_means.quantile(0.90),
        "max_daily_ops": stats["maximum"].max()
    })

# ------------------------------------------------------------
# 2. EXTERNAL EMAIL BYTES
# ------------------------------------------------------------

email_active = activity.loc[
    activity["external_email_count"] > 0
]

email_bytes_stats = (
    email_active.groupby("user")["external_email_bytes"]
    .agg(total="sum", mean="mean", median="median")
)

for stat in ["total", "mean", "median"]:

    col_name = f"email_bytes_{stat}"

    population_profile[col_name] = (
        population_profile["user"].map(
            email_bytes_stats[stat]
        )
    )

population_profile["email_bytes_total"] = (
    population_profile["email_bytes_total"].fillna(0)
)

# ------------------------------------------------------------
# 3. DISPLAY CONSOLIDATED RESULTS
# ------------------------------------------------------------

intensity_summary = pd.DataFrame(intensity_records)

print("=" * 70)
print("CHANNEL ACTIVITY INTENSITY — FUTURE PERIOD")
print("=" * 70)

print("\nPER-USER OPERATION INTENSITY")
print("-" * 70)

print(
    intensity_summary.round(2).to_string(index=False)
)

print("\nEXTERNAL EMAIL DATA VOLUME")
print("-" * 70)

print(
    "Median total bytes per email-active user:",
    f"{email_bytes_stats['total'].median():,.2f}"
)

print(
    "Median of per-user average daily bytes:",
    f"{email_bytes_stats['mean'].median():,.2f}"
)

print("\nVALIDATION: PASS — Active-day counts matched.")

del activity


CHANNEL ACTIVITY INTENSITY — FUTURE PERIOD

PER-USER OPERATION INTENSITY
----------------------------------------------------------------------
channel  users  median_total_ops  median_daily_mean  p25_daily_mean  p75_daily_mean  p90_daily_mean  max_daily_ops
  EMAIL   3737             141.0               2.00            1.22            2.27            2.68           16.0
    USB    719              60.0               1.62            1.38            2.35            5.40           16.0
   HTTP    567              41.0               1.76            1.53            3.83            7.60           18.0

EXTERNAL EMAIL DATA VOLUME
----------------------------------------------------------------------
Median total bytes per email-active user: 36,812,226.00
Median of per-user average daily bytes: 800,825.90

VALIDATION: PASS — Active-day counts matched.



## 4. Temporal Continuity & Gaps

### 4.1 Consecutive Activity Days

**Objective:** Analyze the continuity of natural user activity during the Future Attack Period.

**This analysis will:**
- Calculate the longest consecutive eligible workday streak per user.
- Calculate the longest consecutive active-day streak for Email, USB, and HTTP Upload separately.
- Examine how activity continuity differs across channels and users.
- Distinguish consecutive activity from total active days.

**Consecutive Workday Definition:**
- Monday through Friday are considered scheduled workdays.
- Friday followed by Monday is considered consecutive.
- Missing eligible workdays break the sequence.
- Channel inactivity breaks the corresponding channel streak.

**Important:**
- Only the Future period is analyzed.
- No attack duration or eligibility constraints are imposed.
- The original Baseline remains unchanged.
- Results remain in memory for subsequent analysis.


In [12]:

# ============================================================
# 4.1 CONSECUTIVE ACTIVITY DAYS
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. PREPARE FUTURE ACTIVITY
# ------------------------------------------------------------

temporal = future_activity[
    ["user", "day", "email_active", "usb_active", "http_active"]
].copy()

temporal["day"] = pd.to_datetime(temporal["day"])

temporal = temporal.sort_values(
    ["user", "day"]
).reset_index(drop=True)

# Previous observed day for the same user
previous_user = temporal["user"].shift()
previous_day = temporal["day"].shift()

calendar_gap = (
    temporal["day"] - previous_day
).dt.days

same_user = temporal["user"].eq(previous_user)

# Consecutive scheduled workdays:
# Mon-Thu -> next day, Friday -> Monday
consecutive_date = (
    (calendar_gap == 1)
    | (
        (calendar_gap == 3)
        & (previous_day.dt.dayofweek == 4)
        & (temporal["day"].dt.dayofweek == 0)
    )
)

valid_continuation = same_user & consecutive_date

# ------------------------------------------------------------
# 2. LONGEST ELIGIBLE WORKDAY STREAK
# ------------------------------------------------------------

workday_run_id = (~valid_continuation).cumsum()

workday_runs = (
    temporal.groupby(["user", workday_run_id])
    .size()
)

longest_workday_streak = (
    workday_runs.groupby(level=0).max()
)

population_profile["max_workday_streak"] = (
    population_profile["user"]
    .map(longest_workday_streak)
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 3. LONGEST ACTIVE STREAK PER CHANNEL
# ------------------------------------------------------------

CHANNEL_FLAGS = {
    "Email": "email_active",
    "USB": "usb_active",
    "HTTP": "http_active"
}

summary_records = []

for channel, flag_column in CHANNEL_FLAGS.items():

    active = temporal[flag_column].eq(1)

    previous_active = active.shift(fill_value=False)

    # A new streak starts when:
    # - activity begins
    # - previous day had no channel activity
    # - or the scheduled workday sequence is broken
    new_streak = (
        active
        & ~(previous_active & valid_continuation)
    )

    streak_id = new_streak.cumsum()

    active_runs = pd.DataFrame({
        "user": temporal.loc[active, "user"],
        "streak_id": streak_id.loc[active]
    })

    streak_lengths = (
        active_runs.groupby(["user", "streak_id"])
        .size()
    )

    longest_streak = (
        streak_lengths.groupby(level=0).max()
    )

    output_column = f"max_{channel.lower()}_streak"

    population_profile[output_column] = (
        population_profile["user"]
        .map(longest_streak)
        .fillna(0)
        .astype(int)
    )

    # Summarize only actual users of this channel
    users = population_profile.loc[
        population_profile[f"{channel.lower()}_days"] > 0
    ]

    values = users[output_column]

    summary_records.append({
        "channel": channel,
        "users": len(users),
        "min": values.min(),
        "p25": values.quantile(0.25),
        "median": values.median(),
        "mean": values.mean(),
        "p75": values.quantile(0.75),
        "p90": values.quantile(0.90),
        "max": values.max()
    })

# ------------------------------------------------------------
# 4. ELIGIBLE WORKDAY CONTINUITY
# ------------------------------------------------------------

available = population_profile.loc[
    population_profile["future_workdays"] > 0,
    "max_workday_streak"
]

workday_summary = {
    "channel": "Eligible Workdays",
    "users": len(available),
    "min": available.min(),
    "p25": available.quantile(0.25),
    "median": available.median(),
    "mean": available.mean(),
    "p75": available.quantile(0.75),
    "p90": available.quantile(0.90),
    "max": available.max()
}

summary_records.insert(0, workday_summary)

# ------------------------------------------------------------
# 5. DISPLAY CONSOLIDATED RESULTS
# ------------------------------------------------------------

continuity_summary = pd.DataFrame(summary_records)

print("=" * 75)
print("TEMPORAL CONTINUITY — FUTURE PERIOD")
print("=" * 75)

print("\nLONGEST CONSECUTIVE STREAK PER USER")
print("-" * 75)

print(
    continuity_summary.round(2).to_string(index=False)
)

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

for channel in CHANNEL_FLAGS:

    streak_col = f"max_{channel.lower()}_streak"
    days_col = f"{channel.lower()}_days"

    assert (
        population_profile[streak_col]
        <= population_profile[days_col]
    ).all(), f"{channel}: Streak exceeds active days"

assert (
    population_profile["max_workday_streak"]
    <= population_profile["future_workdays"]
).all(), "Workday streak exceeds available workdays"

print("\nVALIDATION: PASS")


TEMPORAL CONTINUITY — FUTURE PERIOD

LONGEST CONSECUTIVE STREAK PER USER
---------------------------------------------------------------------------
          channel  users  min  p25  median  mean  p75  p90  max
Eligible Workdays   3737    1 79.0    79.0 79.21 79.0 79.0  108
            Email   3737    1  4.0     9.0  9.23 12.0 17.0   61
              USB    719    1  2.0     4.0 11.13  8.0 29.0  107
             HTTP    567    1  2.0     3.0  2.99  4.0  4.0    7

VALIDATION: PASS



### 4.2 Calendar Coverage and Shared Workday Gaps

**Objective:** Investigate whether interruptions in consecutive workday streaks are caused by shared gaps in the Future dataset.

**This analysis will:**
- Examine the number of users with eligible records on each weekday.
- Identify weekdays with no eligible records.
- Identify dates with unusually low user coverage.
- Examine the distribution of longest eligible workday streaks.
- Distinguish dataset-wide interruptions from individual user inactivity.

**Important:**
- Only the Future period is analyzed.
- No attack eligibility constraints are imposed.
- The Baseline remains unchanged.
- No output files are generated.


In [13]:

# ============================================================
# 4.2 CALENDAR COVERAGE AND SHARED WORKDAY GAPS
# ============================================================

import pandas as pd

# Official Future period
FUTURE_START = pd.Timestamp("2011-01-01")
FUTURE_END = pd.Timestamp("2011-06-01")

# All scheduled Monday-Friday dates
expected_weekdays = pd.bdate_range(
    FUTURE_START,
    FUTURE_END
)

# Actual eligible user records per date
daily_coverage = (
    future_activity
    .assign(day=lambda d: pd.to_datetime(d["day"]))
    .groupby("day")["user"]
    .nunique()
    .reindex(expected_weekdays, fill_value=0)
)

daily_coverage.index.name = "day"

# Identify complete gaps
missing_weekdays = daily_coverage[
    daily_coverage == 0
]

# Descriptive coverage threshold only
median_coverage = daily_coverage.median()

low_coverage = daily_coverage[
    (daily_coverage > 0)
    & (daily_coverage < median_coverage * 0.5)
]

# Distribution of longest eligible workday streaks
streak_distribution = (
    population_profile.loc[
        population_profile["future_workdays"] > 0,
        "max_workday_streak"
    ]
    .value_counts()
    .head(10)
)

print("=" * 65)
print("FUTURE CALENDAR COVERAGE")
print("=" * 65)

print(f"Expected weekdays       : {len(expected_weekdays)}")
print(f"Weekdays with records   : {(daily_coverage > 0).sum()}")
print(f"Weekdays without records: {len(missing_weekdays)}")
print(f"Median users per weekday: {median_coverage:,.0f}")

print("\nMISSING WEEKDAYS")
print("-" * 65)

if missing_weekdays.empty:
    print("None")
else:
    for day in missing_weekdays.index:
        print(day.strftime("%Y-%m-%d (%A)"))

print("\nLOW-COVERAGE WEEKDAYS")
print("-" * 65)

if low_coverage.empty:
    print("None")
else:
    print(low_coverage.to_string())

print("\nMOST COMMON LONGEST WORKDAY STREAKS")
print("-" * 65)
print(streak_distribution.to_string())


FUTURE CALENDAR COVERAGE
Expected weekdays       : 108
Weekdays with records   : 108
Weekdays without records: 0
Median users per weekday: 3,680

MISSING WEEKDAYS
-----------------------------------------------------------------
None

LOW-COVERAGE WEEKDAYS
-----------------------------------------------------------------
day
2011-04-22    160
2011-05-30    180
2011-06-01      7

MOST COMMON LONGEST WORKDAY STREAKS
-----------------------------------------------------------------
max_workday_streak
79     3479
107      82
105      75
61        5
50        5
48        4
30        4
20        4
57        3
37        3



### 4.3 Shared Workday Gap Investigation

**Objective:** Investigate unusually low user coverage on specific dates during the Future period.

This analysis will:

- Compare user coverage before, during, and after the identified low-coverage dates.
- Compare eligible workdays in User-Day against Behavior Features.
- Determine whether coverage differences occur before or during feature generation.
- Examine the relationship between April 22 and the common 79-day workday streak.

**Important:** Low coverage does not automatically indicate missing data or abnormal user behavior. No baseline data will be modified.


In [14]:

# ============================================================
# 4.3 SHARED WORKDAY GAP INVESTIGATION
# ============================================================

import pandas as pd

# Dates surrounding unusual coverage drops
check_dates = pd.bdate_range(
    "2011-04-20", "2011-04-26"
).union(
    pd.bdate_range("2011-05-26", "2011-06-01")
)

# Load original integrated daily records
daily = pd.read_parquet(
    DATASETS["user_day"],
    columns=["user", "day", "is_weekend", "logon_count"]
)

daily["day"] = pd.to_datetime(daily["day"])

# Restrict to benign baseline users and selected dates
benign_users = set(population_profile["user"])

daily = daily.loc[
    daily["user"].isin(benign_users)
    & daily["day"].isin(check_dates)
]

eligible = daily.loc[
    (daily["is_weekend"] == 0)
    & (daily["logon_count"] > 0)
]

# Compare against available Future features
feature_dates = future_activity.loc[
    pd.to_datetime(future_activity["day"]).isin(check_dates)
].copy()

feature_dates["day"] = pd.to_datetime(
    feature_dates["day"]
)

coverage = pd.DataFrame(index=check_dates)
coverage.index.name = "day"

coverage["user_day_rows"] = (
    daily.groupby("day").size()
)

coverage["eligible_workdays"] = (
    eligible.groupby("day").size()
)

coverage["feature_rows"] = (
    feature_dates.groupby("day").size()
)

coverage = coverage.fillna(0).astype(int)

coverage["difference"] = (
    coverage["eligible_workdays"]
    - coverage["feature_rows"]
)

coverage["coverage_pct"] = (
    coverage["feature_rows"]
    / (population_profile["future_workdays"] > 0).sum()
    * 100
).round(2)

print("=" * 75)
print("SHARED WORKDAY GAP INVESTIGATION")
print("=" * 75)

print(coverage.to_string())

# Investigate users with a 79-day maximum streak
users_79 = set(
    population_profile.loc[
        population_profile["max_workday_streak"] == 79,
        "user"
    ]
)

present_apr22 = set(
    future_activity.loc[
        pd.to_datetime(future_activity["day"])
        == pd.Timestamp("2011-04-22"),
        "user"
    ]
)

print("\n79-DAY STREAK INVESTIGATION")
print("-" * 75)

print("Users with max streak = 79:", len(users_79))
print(
    "Of these, users absent on April 22:",
    len(users_79 - present_apr22)
)

print("\nSource-to-feature mismatches:", coverage["difference"].abs().sum())

print(
    "VALIDATION:",
    "PASS" if (coverage["difference"] == 0).all()
    else "REVIEW"
)


SHARED WORKDAY GAP INVESTIGATION
            user_day_rows  eligible_workdays  feature_rows  difference  coverage_pct
day                                                                                 
2011-04-20           3657               3657          3657           0         97.86
2011-04-21           3657               3657          3657           0         97.86
2011-04-22            165                160           160           0          4.28
2011-04-25           3654               3654          3654           0         97.78
2011-04-26           3654               3654          3654           0         97.78
2011-05-26           3636               3636          3636           0         97.30
2011-05-27           3636               3636          3636           0         97.30
2011-05-30            181                180           180           0          4.82
2011-05-31           3636               3636          3636           0         97.30
2011-06-01             13       


### 4.4 Shared Activity Drop Investigation

**Objective:** Investigate whether the sharp drops in user coverage on April 22, May 30, and June 1, 2011 affect all activity sources or only logon activity.

**This analysis will:**
- Compare the three low-coverage dates with surrounding workdays.
- Count users with available daily records.
- Examine logon, email, file, and HTTP activity.
- Distinguish general event coverage drops from missing eligible logon activity.
- Determine whether the interruptions are shared across multiple channels.

**Important:**
- Only the Future period and benign baseline users are analyzed.
- Results are based on processed Data Engineering records.
- The original raw-event cause cannot be confirmed from these records alone.
- No original data is modified.


In [15]:

# ============================================================
# 4.4 SHARED ACTIVITY DROP INVESTIGATION
# ============================================================

import pandas as pd

# Dates selected for comparison
CHECK_DATES = pd.to_datetime([
    "2011-04-20",
    "2011-04-21",
    "2011-04-22",
    "2011-04-25",
    "2011-04-26",
    "2011-05-26",
    "2011-05-27",
    "2011-05-30",
    "2011-05-31",
    "2011-06-01"
])

# Load only required measurements
columns = [
    "user", "day", "is_weekend", "logon_count",
    "email_event_count", "external_email_count",
    "file_event_count", "usb_copy_count",
    "http_event_count", "www_upload_count"
]

daily = pd.read_parquet(
    DATASETS["user_day"],
    columns=columns
)

daily["day"] = pd.to_datetime(daily["day"])

# Restrict to benign users and selected Future dates
daily = daily.loc[
    daily["user"].isin(population_profile["user"])
    & daily["day"].isin(CHECK_DATES)
].copy()

# Activity indicators
daily["eligible"] = (
    (daily["is_weekend"] == 0)
    & (daily["logon_count"] > 0)
)

ACTIVITY_FLAGS = {
    "logon_users": "logon_count",
    "email_users": "email_event_count",
    "file_users": "file_event_count",
    "http_users": "http_event_count",
    "external_email_users": "external_email_count",
    "usb_copy_users": "usb_copy_count",
    "http_upload_users": "www_upload_count"
}

for flag, column in ACTIVITY_FLAGS.items():
    daily[flag] = (daily[column] > 0).astype(int)

# Aggregate by date
coverage = daily.groupby("day").agg(
    user_day_records=("user", "nunique"),
    eligible_users=("eligible", "sum"),
    **{
        flag: (flag, "sum")
        for flag in ACTIVITY_FLAGS
    }
)

coverage = (
    coverage.reindex(CHECK_DATES, fill_value=0)
    .rename_axis("day")
    .reset_index()
)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("SHARED ACTIVITY DROP INVESTIGATION")
print("=" * 70)

print("\n1. RECORD AND LOGON COVERAGE")
print("-" * 70)

print(
    coverage[
        ["day", "user_day_records",
         "eligible_users", "logon_users"]
    ].to_string(index=False)
)

print("\n2. GENERAL CHANNEL ACTIVITY")
print("-" * 70)

print(
    coverage[
        ["day", "email_users",
         "file_users", "http_users"]
    ].to_string(index=False)
)

print("\n3. TARGET EXFILTRATION CHANNELS")
print("-" * 70)

print(
    coverage[
        ["day", "external_email_users",
         "usb_copy_users", "http_upload_users"]
    ].to_string(index=False)
)

# Save results in memory only
activity_drop_summary = coverage.copy()

print("\nSTATUS: Investigation completed.")


SHARED ACTIVITY DROP INVESTIGATION

1. RECORD AND LOGON COVERAGE
----------------------------------------------------------------------
       day  user_day_records  eligible_users  logon_users
2011-04-20              3657            3657         3657
2011-04-21              3657            3657         3657
2011-04-22               165             160          160
2011-04-25              3654            3654         3654
2011-04-26              3654            3654         3654
2011-05-26              3636            3636         3636
2011-05-27              3636            3636         3636
2011-05-30               181             180          180
2011-05-31              3636            3636         3636
2011-06-01                13               7            7

2. GENERAL CHANNEL ACTIVITY
----------------------------------------------------------------------
       day  email_users  file_users  http_users
2011-04-20         3640         851        3656
2011-04-21         3623       


### Findings — Shared Activity Drops

The analysis identified three dates with unusually low user activity: April 22, May 30, and June 1, 2011.

The reductions affected Logon, Email, File, and HTTP activity, indicating a dataset-wide activity drop rather than a channel-specific issue.

April 22 coincided with Good Friday, and May 30 with Memorial Day, suggesting possible holiday-related inactivity. June 1 was the final date of the available dataset and showed very limited records.

These explanations remain hypotheses because the original raw-event collection process has not been independently verified.

**Decision:** Preserve all original records and account for shared calendar interruptions when interpreting user activity continuity. Attack scheduling will follow the existing eligible-workday definition.



### 4.5 Channel Activity Gaps

**Objective:** Analyze the gaps between consecutive active days in Email, USB, and HTTP Upload during the Future Attack Period.

**This analysis will:**
- Calculate inactive eligible workdays between consecutive channel activities.
- Compare eligible-workday gaps with calendar-workday gaps.
- Measure median, P90, and maximum gap lengths.
- Examine differences in activity gaps across users.
- Identify the impact of missing eligible workdays on gap measurements.

**Gap Definitions:**
- **Eligible Workday Gap:** Number of eligible user-workdays between two active channel days.
- **Calendar Workday Gap:** Number of Monday–Friday dates between two active channel days.
- **Zero Gap:** Two activities occur on consecutive workdays under the corresponding definition.

**Important:**
- Only natural activity during the Future period is analyzed.
- The Baseline remains unchanged.
- Users with fewer than two active days have no measurable between-activity gaps.
- No attack eligibility thresholds are applied.
- Results remain in memory.


In [16]:

# ============================================================
# 4.5 CHANNEL ACTIVITY GAPS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PREPARE FUTURE ACTIVITY
# ------------------------------------------------------------

temporal = future_activity[
    ["user", "day", "email_active", "usb_active", "http_active"]
].copy()

temporal["day"] = pd.to_datetime(temporal["day"])

temporal = temporal.sort_values(
    ["user", "day"]
).reset_index(drop=True)

# Position within each user's eligible workdays
temporal["eligible_position"] = (
    temporal.groupby("user").cumcount()
)

CHANNEL_FLAGS = {
    "email": "email_active",
    "usb": "usb_active",
    "http": "http_active"
}

gap_summary_records = []

# ------------------------------------------------------------
# 2. CALCULATE CHANNEL GAPS
# ------------------------------------------------------------

for channel, flag in CHANNEL_FLAGS.items():

    active = temporal.loc[
        temporal[flag] == 1,
        ["user", "day", "eligible_position"]
    ].copy()

    # Previous active day for the same user
    active["previous_day"] = (
        active.groupby("user")["day"].shift()
    )

    active["previous_position"] = (
        active.groupby("user")["eligible_position"].shift()
    )

    # Consecutive pairs of observed channel activity
    gaps = active.dropna(
        subset=["previous_day"]
    ).copy()

    # Eligible workdays between two active days
    gaps["gap_eligible_wd"] = (
        gaps["eligible_position"]
        - gaps["previous_position"]
        - 1
    ).astype(int)

    # Calendar weekdays between two active days
    gaps["gap_calendar_wd"] = (
        np.busday_count(
            gaps["previous_day"].to_numpy(
                dtype="datetime64[D]"
            ),
            gaps["day"].to_numpy(
                dtype="datetime64[D]"
            )
        ) - 1
    )

    # --------------------------------------------------------
    # 3. PER-USER GAP STATISTICS
    # --------------------------------------------------------

    user_gaps = gaps.groupby("user").agg(
        gap_pairs=("gap_eligible_wd", "size"),
        gap_median=("gap_eligible_wd", "median"),
        gap_mean=("gap_eligible_wd", "mean"),
        gap_p90=("gap_eligible_wd", lambda x: x.quantile(0.90)),
        gap_max=("gap_eligible_wd", "max"),
        calendar_gap_median=("gap_calendar_wd", "median"),
        calendar_gap_max=("gap_calendar_wd", "max")
    )

    for stat in user_gaps.columns:

        col_name = f"{channel}_{stat}"

        population_profile[col_name] = (
            population_profile["user"].map(user_gaps[stat])
        )

    population_profile[f"{channel}_gap_pairs"] = (
        population_profile[f"{channel}_gap_pairs"]
        .fillna(0)
        .astype(int)
    )

    # --------------------------------------------------------
    # 4. VALIDATION
    # --------------------------------------------------------

    expected_pairs = (
        population_profile[f"{channel}_days"]
        .sub(1)
        .clip(lower=0)
        .sum()
    )

    assert len(gaps) == expected_pairs, (
        f"{channel}: Incorrect number of gap pairs"
    )

    assert (gaps["gap_eligible_wd"] >= 0).all()
    assert (
        gaps["gap_calendar_wd"] >= gaps["gap_eligible_wd"]
    ).all()

    # --------------------------------------------------------
    # 5. CHANNEL SUMMARY
    # --------------------------------------------------------

    gap_summary_records.append({
        "channel": channel.upper(),
        "users_with_2plus_days": len(user_gaps),
        "gap_pairs": len(gaps),
        "median_eligible_gap": gaps["gap_eligible_wd"].median(),
        "p90_eligible_gap": gaps["gap_eligible_wd"].quantile(0.90),
        "max_eligible_gap": gaps["gap_eligible_wd"].max(),
        "median_user_gap": user_gaps["gap_median"].median(),
        "median_calendar_gap": gaps["gap_calendar_wd"].median(),
        "max_calendar_gap": gaps["gap_calendar_wd"].max(),
        "gaps_crossing_ineligible_days": int(
            (gaps["gap_calendar_wd"] > gaps["gap_eligible_wd"]).sum()
        )
    })

# ------------------------------------------------------------
# 6. DISPLAY CONSOLIDATED RESULTS
# ------------------------------------------------------------

gap_summary = pd.DataFrame(gap_summary_records)

print("=" * 70)
print("CHANNEL ACTIVITY GAPS — FUTURE PERIOD")
print("=" * 70)

print("\nELIGIBLE WORKDAY GAPS")
print("-" * 70)

print(
    gap_summary[
        [
            "channel",
            "users_with_2plus_days",
            "gap_pairs",
            "median_eligible_gap",
            "p90_eligible_gap",
            "max_eligible_gap",
            "median_user_gap"
        ]
    ].round(2).to_string(index=False)
)

print("\nCALENDAR WORKDAY GAPS")
print("-" * 70)

print(
    gap_summary[
        [
            "channel",
            "median_calendar_gap",
            "max_calendar_gap",
            "gaps_crossing_ineligible_days"
        ]
    ].round(2).to_string(index=False)
)

print("\nVALIDATION: PASS")


CHANNEL ACTIVITY GAPS — FUTURE PERIOD

ELIGIBLE WORKDAY GAPS
----------------------------------------------------------------------
channel  users_with_2plus_days  gap_pairs  median_eligible_gap  p90_eligible_gap  max_eligible_gap  median_user_gap
  EMAIL                   3733     218102                  0.0               2.0                56              0.0
    USB                    715      29299                  0.0               4.0               100              1.0
   HTTP                    566      13331                  2.0               8.0                41              2.0

CALENDAR WORKDAY GAPS
----------------------------------------------------------------------
channel  median_calendar_gap  max_calendar_gap  gaps_crossing_ineligible_days
  EMAIL                  0.0                57                           5364
    USB                  0.0               101                            851
   HTTP                  2.0                41                            53


### 4.6 Natural Channel Transitions

**Objective:** Analyze natural channel-switching behavior across eligible user-workdays during the Future Attack Period.

**This analysis will:**
- Identify daily combinations of Email, USB, and HTTP Upload.
- Measure direct switches between different single-channel days.
- Distinguish direct switching from changes involving multi-channel overlap.
- Compare transitions on adjacent eligible workdays with transitions between successive active days.
- Count switches per user and identify the most frequent transition patterns.

**Definitions:**
- **Direct Switch:** A transition from one single-channel day to a different single-channel day (e.g., Email → USB).
- **Channel Set Change:** A change in the combination of active channels (e.g., Email → Email + USB).
- **Adjacent Eligible Transition:** Two successive eligible workdays for the same user, even if separated by calendar dates.
- **Successive Active-Day Transition:** Two successive days with target-channel activity, allowing inactive eligible workdays between them.

**Important:**
- Only natural Future-period activity is analyzed.
- No attack-switching requirements are imposed.
- The Baseline remains unchanged.
- Results remain in memory.


In [17]:

# ============================================================
# 4.6 NATURAL CHANNEL TRANSITIONS
# ============================================================

import pandas as pd

# 1. Prepare chronological daily activity
transitions = future_activity[
    ["user", "day", "email_active", "usb_active", "http_active"]
].copy()

transitions["day"] = pd.to_datetime(transitions["day"])

transitions = transitions.sort_values(
    ["user", "day"]
).reset_index(drop=True)

assert not transitions.duplicated(["user", "day"]).any()

# E = Email, U = USB, H = HTTP, "" = No target channel
transitions["state"] = (
    transitions["email_active"].map({0: "", 1: "E"})
    + transitions["usb_active"].map({0: "", 1: "U"})
    + transitions["http_active"].map({0: "", 1: "H"})
)

transitions["eligible_position"] = (
    transitions.groupby("user").cumcount()
)

# Reusable transition classification
def classify_transitions(pairs):

    old = pairs["previous_state"]
    new = pairs["state"]

    both_active = old.ne("") & new.ne("")

    pairs["active_set_change"] = (
        both_active & old.ne(new)
    )

    pairs["direct_switch"] = (
        both_active
        & old.str.len().eq(1)
        & new.str.len().eq(1)
        & old.ne(new)
    )

    pairs["overlap_change"] = (
        pairs["active_set_change"]
        & ~pairs["direct_switch"]
    )

    return pairs

# ------------------------------------------------------------
# 2. ADJACENT ELIGIBLE WORKDAY TRANSITIONS
# ------------------------------------------------------------

transitions["previous_state"] = (
    transitions.groupby("user")["state"].shift()
)

adjacent = transitions.loc[
    transitions["previous_state"].notna()
].copy()

adjacent = classify_transitions(adjacent)

# ------------------------------------------------------------
# 3. SUCCESSIVE ACTIVE-DAY TRANSITIONS
# ------------------------------------------------------------

active = transitions.loc[
    transitions["state"] != ""
].copy()

active["previous_state"] = (
    active.groupby("user")["state"].shift()
)

active["previous_position"] = (
    active.groupby("user")["eligible_position"].shift()
)

active_pairs = active.loc[
    active["previous_state"].notna()
].copy()

active_pairs = classify_transitions(active_pairs)

active_pairs["inactive_days_between"] = (
    active_pairs["eligible_position"]
    - active_pairs["previous_position"]
    - 1
).astype(int)

# ------------------------------------------------------------
# 4. PER-USER SWITCHING PROFILE
# ------------------------------------------------------------

switch_metrics = {
    "adjacent_direct_switches": (
        adjacent.groupby("user")["direct_switch"].sum()
    ),
    "adjacent_set_changes": (
        adjacent.groupby("user")["active_set_change"].sum()
    ),
    "active_day_direct_switches": (
        active_pairs.groupby("user")["direct_switch"].sum()
    ),
    "active_day_overlap_changes": (
        active_pairs.groupby("user")["overlap_change"].sum()
    )
}

for column, values in switch_metrics.items():

    population_profile[column] = (
        population_profile["user"]
        .map(values)
        .fillna(0)
        .astype(int)
    )

# ------------------------------------------------------------
# 5. CONSOLIDATED SUMMARY
# ------------------------------------------------------------

print("=" * 72)
print("NATURAL CHANNEL TRANSITIONS — FUTURE PERIOD")
print("=" * 72)

print("\nADJACENT ELIGIBLE WORKDAYS")
print("-" * 72)

print(f"Total day pairs          : {len(adjacent):,}")
print(f"Active channel-set changes: {adjacent['active_set_change'].sum():,}")
print(f"Direct channel switches  : {adjacent['direct_switch'].sum():,}")
print(f"Changes involving overlap: {adjacent['overlap_change'].sum():,}")

print("\nSUCCESSIVE ACTIVE DAYS")
print("-" * 72)

print(f"Active-day pairs         : {len(active_pairs):,}")
print(f"Active channel-set changes: {active_pairs['active_set_change'].sum():,}")
print(f"Direct channel switches  : {active_pairs['direct_switch'].sum():,}")
print(f"Changes involving overlap: {active_pairs['overlap_change'].sum():,}")

print(
    "Direct switches with inactive days between:",
    int((
        active_pairs["direct_switch"]
        & (active_pairs["inactive_days_between"] > 0)
    ).sum())
)

# ------------------------------------------------------------
# 6. USER-LEVEL DISTRIBUTION
# ------------------------------------------------------------

users = population_profile.loc[
    population_profile["future_workdays"] > 0
]

print("\nUSERS WITH NATURAL SWITCHING")
print("-" * 72)

for column in switch_metrics:
    print(
        f"{column:<32}",
        f"{(users[column] > 0).sum():,} users"
    )

print("\nDIRECT SWITCHES PER USER")
print("-" * 72)

print(
    users[
        ["adjacent_direct_switches", "active_day_direct_switches"]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    ).round(2).to_string()
)

# ------------------------------------------------------------
# 7. MOST COMMON CHANGED CHANNEL COMBINATIONS
# ------------------------------------------------------------

changed_pairs = active_pairs.loc[
    active_pairs["active_set_change"]
]

top_transitions = (
    changed_pairs
    .groupby(["previous_state", "state"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print("\nTOP 10 ACTIVE CHANNEL TRANSITIONS")
print("-" * 72)

print(top_transitions.to_string())

# ------------------------------------------------------------
# 8. VALIDATION
# ------------------------------------------------------------

assert (
    adjacent["direct_switch"]
    <= adjacent["active_set_change"]
).all()

assert (
    active_pairs["direct_switch"]
    <= active_pairs["active_set_change"]
).all()

print("\nVALIDATION: PASS")


NATURAL CHANNEL TRANSITIONS — FUTURE PERIOD

ADJACENT ELIGIBLE WORKDAYS
------------------------------------------------------------------------
Total day pairs          : 383,425
Active channel-set changes: 31,558
Direct channel switches  : 6,953
Changes involving overlap: 24,605

SUCCESSIVE ACTIVE DAYS
------------------------------------------------------------------------
Active-day pairs         : 235,983
Active channel-set changes: 40,164
Direct channel switches  : 10,148
Changes involving overlap: 30,016
Direct switches with inactive days between: 3195

USERS WITH NATURAL SWITCHING
------------------------------------------------------------------------
adjacent_direct_switches         919 users
adjacent_set_changes             994 users
active_day_direct_switches       928 users
active_day_overlap_changes       985 users

DIRECT SWITCHES PER USER
------------------------------------------------------------------------
       adjacent_direct_switches  active_day_direct_switches



### 4.7 Channel Activity Distribution Over Time

**Objective:** Examine how natural Email, USB, and HTTP Upload activity is distributed over time for each user during the Future Attack Period.

**This analysis will:**
- Count active days per channel for each user in every month.
- Count the number of months with activity in each channel.
- Identify the first and last active dates for each channel.
- Identify the month with the highest activity for each user and channel.
- Calculate the percentage of total channel activity days concentrated in the most active month.
- Compare monthly channel availability across the population.

**Analysis Scope:**
- Future Period: January 1 – June 1, 2011.
- Only eligible user-workdays are analyzed.
- Users without activity in a channel remain included.
- The original Baseline remains unchanged.
- No attack eligibility or switching constraints are applied.

**Expected Outputs:**
- `01_channel_monthly_activity.parquet`
- `01_channel_temporal_profile.parquet`

**Important:** Monthly activity concentration is based on the number of active days, not the number of operations. June 2011 contains only the final day of the dataset and must not be interpreted as a complete month.


In [18]:

# ============================================================
# 4.7 CHANNEL ACTIVITY DISTRIBUTION OVER TIME
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PREPARE FUTURE ACTIVITY
# ------------------------------------------------------------

channels = ["email", "usb", "http"]

future_start = pd.Timestamp("2011-01-01")
future_end = pd.Timestamp("2011-06-01")

months = pd.period_range(
    future_start, future_end, freq="M"
).astype(str).tolist()

source = future_activity[
    ["user", "day"] + [f"{c}_active" for c in channels]
].copy()

source["day"] = pd.to_datetime(source["day"])

assert source["day"].between(future_start, future_end).all()
assert not source.duplicated(["user", "day"]).any()
assert source[["user", "day"]].notna().all().all()

source["month"] = source["day"].dt.strftime("%Y-%m")

# ------------------------------------------------------------
# 2. MONTHLY ACTIVITY PER USER
# ------------------------------------------------------------

monthly = (
    source.groupby(["user", "month"])
    .agg(
        eligible_workdays=("day", "size"),
        email_days=("email_active", "sum"),
        usb_days=("usb_active", "sum"),
        http_days=("http_active", "sum")
    )
)

# Include all baseline users and all months,
# including months with no eligible workdays.
full_index = pd.MultiIndex.from_product(
    [population_profile["user"], months],
    names=["user", "month"]
)

monthly = (
    monthly.reindex(full_index, fill_value=0)
    .reset_index()
)

count_cols = [
    "eligible_workdays",
    "email_days",
    "usb_days",
    "http_days"
]

monthly[count_cols] = monthly[count_cols].astype(int)

# ------------------------------------------------------------
# 3. PER-USER TEMPORAL PROFILE
# ------------------------------------------------------------

temporal_profile = pd.DataFrame({
    "user": population_profile["user"].to_numpy()
})

for channel in channels:

    days_col = f"{channel}_days"
    flag_col = f"{channel}_active"

    monthly_counts = monthly.groupby("user")[days_col]

    total_days = temporal_profile["user"].map(
        monthly_counts.sum()
    ).fillna(0)

    active_months = (
        monthly.loc[monthly[days_col] > 0]
        .groupby("user").size()
    )

    # Highest-activity month; earliest month wins ties.
    peak = (
        monthly.loc[monthly[days_col] > 0]
        .sort_values(
            ["user", days_col, "month"],
            ascending=[True, False, True]
        )
        .drop_duplicates("user")
        .set_index("user")
    )

    # First and last observed active dates
    active_dates = (
        source.loc[source[flag_col] == 1]
        .groupby("user")["day"]
        .agg(["min", "max"])
    )

    temporal_profile[f"{channel}_active_months"] = (
        temporal_profile["user"]
        .map(active_months)
        .fillna(0).astype(int)
    )

    temporal_profile[f"{channel}_first_active_day"] = (
        temporal_profile["user"].map(active_dates["min"])
    )

    temporal_profile[f"{channel}_last_active_day"] = (
        temporal_profile["user"].map(active_dates["max"])
    )

    temporal_profile[f"{channel}_peak_month"] = (
        temporal_profile["user"].map(peak["month"])
    )

    peak_days = (
        temporal_profile["user"]
        .map(peak[days_col])
        .fillna(0)
        .astype(int)
    )

    temporal_profile[f"{channel}_peak_month_days"] = peak_days

    temporal_profile[f"{channel}_peak_month_share_pct"] = (
        peak_days
        .div(total_days.replace(0, np.nan))
        .mul(100)
        .round(2)
    )

# ------------------------------------------------------------
# 4. VALIDATE AGAINST PREVIOUS ANALYSES
# ------------------------------------------------------------

for column in count_cols:

    monthly_total = (
        monthly.groupby("user")[column].sum()
    )

    original_column = (
        "future_workdays"
        if column == "eligible_workdays"
        else column
    )

    expected = (
        population_profile["user"]
        .map(monthly_total)
        .fillna(0)
        .astype(int)
    )

    assert expected.equals(
        population_profile[original_column].reset_index(drop=True)
    ), f"Mismatch detected: {column}"

assert len(monthly) == len(population_profile) * len(months)
assert temporal_profile["user"].is_unique

# Add new properties to main population profile
indexed_profile = temporal_profile.set_index("user")

for column in temporal_profile.columns.drop("user"):
    population_profile[column] = (
        population_profile["user"].map(indexed_profile[column])
    )

# ------------------------------------------------------------
# 5. CONSOLIDATED POPULATION RESULTS
# ------------------------------------------------------------

monthly_summary = (
    monthly.groupby("month")[count_cols]
    .sum()
    .reset_index()
)

print("=" * 75)
print("CHANNEL ACTIVITY DISTRIBUTION OVER TIME")
print("=" * 75)

print("\nMONTHLY ACTIVITY — ALL USERS")
print("-" * 75)
print(monthly_summary.to_string(index=False))

print("\nPER-USER TEMPORAL CHARACTERISTICS")
print("-" * 75)

for channel in channels:

    users = temporal_profile.loc[
        temporal_profile[f"{channel}_active_months"] > 0
    ]

    print(f"\n{channel.upper()}")
    print(f"Active users           : {len(users):,}")
    print(
        "Median active months   :",
        users[f"{channel}_active_months"].median()
    )
    print(
        "Median peak-month share:",
        f"{users[f'{channel}_peak_month_share_pct'].median():.2f}%"
    )

# ------------------------------------------------------------
# 6. SAVE IMPORTANT RESULTS
# ------------------------------------------------------------

monthly_path = (
    REPORT_DIR / "01_channel_monthly_activity.parquet"
)

profile_path = (
    REPORT_DIR / "01_channel_temporal_profile.parquet"
)

monthly.to_parquet(monthly_path, index=False)
temporal_profile.to_parquet(profile_path, index=False)

print("\n" + "=" * 75)
print("VALIDATION: PASS")
print(f"Monthly records : {len(monthly):,}")
print(f"User profiles   : {len(temporal_profile):,}")
print("\nSAVED OUTPUTS:")
print(monthly_path.name)
print(profile_path.name)


CHANNEL ACTIVITY DISTRIBUTION OVER TIME

MONTHLY ACTIVITY — ALL USERS
---------------------------------------------------------------------------
  month  eligible_workdays  email_days  usb_days  http_days
2011-01              78299       45437      6055       2279
2011-02              74127       40898      5754       2843
2011-03              84700       48441      6623       3006
2011-04              73387       43557      5662       2372
2011-05              76642       43506      5924       3398
2011-06                  7           0         0          0

PER-USER TEMPORAL CHARACTERISTICS
---------------------------------------------------------------------------

EMAIL
Active users           : 3,737
Median active months   : 5.0
Median peak-month share: 24.10%

USB
Active users           : 719
Median active months   : 5.0
Median peak-month share: 26.67%

HTTP
Active users           : 567
Median active months   : 5.0
Median peak-month share: 28.57%

VALIDATION: PASS
Monthly records


## 5. Final User Behavioral Profile

### 5.1 Complete and Validate User Features

**Objective:** Build a consolidated user-level profile from all completed Future-period analyses.

**Additional Features:**
- Seven exact daily channel combinations: E, U, H, EU, EH, UH, EUH.
- Three historical channel-baseline availability flags.
- First and last eligible Future workday.
- Observed channel set and months with target-channel activity.

**Corrections:**
- Explicitly distinguish users with no eligible Future workdays.
- Rename streak features to clarify that continuity is measured across consecutive calendar weekdays.
- Preserve undefined statistics as missing values rather than replacing them with zero.

**Validation:**
- Verify all daily channel combinations against existing activity counts.
- Verify that all 3,995 baseline users are retained.
- Verify baseline flags using the existing historical baseline metrics.
- Confirm that previously computed frequency, gap, transition, and monthly features remain available.

**Rules:**
- Historical Baseline is read-only.
- All behavioral features use the Future Period only.
- No predefined attack pattern or scenario eligibility criteria are introduced.
- No output is saved until all validations pass.


In [19]:

# ============================================================
# 5.1 COMPLETE AND VALIDATE USER FEATURES
# ============================================================

import pandas as pd
import numpy as np

final_profile = population_profile.copy().reset_index(drop=True)
original_columns = set(final_profile.columns)

# ------------------------------------------------------------
# 1. EXACT DAILY CHANNEL COMBINATIONS
# ------------------------------------------------------------

activity = future_activity[
    ["user", "day", "email_active", "usb_active", "http_active"]
].copy()

assert not activity.duplicated(["user", "day"]).any()
assert activity["user"].isin(final_profile["user"]).all()

for channel in ["email", "usb", "http"]:
    assert activity[f"{channel}_active"].isin([0, 1]).all()

activity["channel_state"] = (
    activity["email_active"].map({0: "", 1: "E"})
    + activity["usb_active"].map({0: "", 1: "U"})
    + activity["http_active"].map({0: "", 1: "H"})
)

channel_states = ["E", "U", "H", "EU", "EH", "UH", "EUH"]

state_counts = (
    activity.groupby(["user", "channel_state"])
    .size()
    .unstack(fill_value=0)
)

for state in channel_states:
    values = (
        state_counts[state]
        if state in state_counts.columns
        else pd.Series(dtype=int)
    )

    final_profile[f"days_{state}"] = (
        final_profile["user"]
        .map(values)
        .fillna(0)
        .astype(int)
    )

# Check against previously computed channel totals
assert (
    final_profile["email_days"]
    == final_profile[
        ["days_E", "days_EU", "days_EH", "days_EUH"]
    ].sum(axis=1)
).all()

assert (
    final_profile["usb_days"]
    == final_profile[
        ["days_U", "days_EU", "days_UH", "days_EUH"]
    ].sum(axis=1)
).all()

assert (
    final_profile["http_days"]
    == final_profile[
        ["days_H", "days_EH", "days_UH", "days_EUH"]
    ].sum(axis=1)
).all()

assert (
    final_profile["days_1ch"]
    == final_profile[["days_E", "days_U", "days_H"]].sum(axis=1)
).all()

assert (
    final_profile["days_2ch"]
    == final_profile[["days_EU", "days_EH", "days_UH"]].sum(axis=1)
).all()

assert (
    final_profile["days_3ch"]
    == final_profile["days_EUH"]
).all()

assert (
    final_profile["future_workdays"]
    == final_profile["days_0ch"]
       + final_profile[channel_states_to_columns := [
           f"days_{s}" for s in channel_states
       ]].sum(axis=1)
).all()

# ------------------------------------------------------------
# 2. HISTORICAL CHANNEL BASELINE AVAILABILITY
# ------------------------------------------------------------

baseline_fields = {
    "email": (
        "email_active_days",
        "email_count_p50",
        "email_count_p95",
        "email_count_mad"
    ),
    "usb": (
        "usb_active_days",
        "usb_copy_p50",
        "usb_copy_p95",
        "usb_copy_mad"
    ),
    "http": (
        "http_upload_active_days",
        "http_upload_p50",
        "http_upload_p95",
        "http_upload_mad"
    )
}

required_baseline_columns = ["user"] + [
    col
    for fields in baseline_fields.values()
    for col in fields
]

baseline_reference = pd.read_parquet(
    DATA_ROOT / "features" / "user_baseline_r62_v1.parquet",
    columns=required_baseline_columns
)

assert baseline_reference["user"].is_unique
assert set(final_profile["user"]) == set(baseline_reference["user"])

baseline_indexed = baseline_reference.set_index("user")

for channel, fields in baseline_fields.items():

    active_days, p50, p95, mad = fields

    # Valid precomputed channel baseline:
    # historical activity + usable P50/P95/MAD.
    valid = (
        baseline_indexed[active_days].gt(0)
        & baseline_indexed[p50].gt(0)
        & baseline_indexed[p95].gt(0)
        & baseline_indexed[p95].ge(baseline_indexed[p50])
        & baseline_indexed[mad].notna()
    ).fillna(False)

    final_profile[f"has_{channel}_channel_baseline"] = (
        final_profile["user"]
        .map(valid)
        .fillna(False)
        .astype(bool)
    )

# ------------------------------------------------------------
# 3. FIRST AND LAST ELIGIBLE FUTURE WORKDAY
# ------------------------------------------------------------

activity["day"] = pd.to_datetime(activity["day"])

future_bounds = activity.groupby("user")["day"].agg(
    first="min",
    last="max"
)

final_profile["future_first_eligible_day"] = (
    final_profile["user"].map(future_bounds["first"])
)

final_profile["future_last_eligible_day"] = (
    final_profile["user"].map(future_bounds["last"])
)

has_future = final_profile["future_workdays"].gt(0)

assert final_profile.loc[
    has_future, "future_first_eligible_day"
].notna().all()

assert final_profile.loc[
    ~has_future, "future_first_eligible_day"
].isna().all()

assert final_profile.loc[
    ~has_future, "future_last_eligible_day"
].isna().all()

# ------------------------------------------------------------
# 4. OBSERVED CHANNEL SET
# ------------------------------------------------------------

final_profile["observed_channel_set"] = (
    final_profile["email_days"].gt(0).map({True: "E", False: ""})
    + final_profile["usb_days"].gt(0).map({True: "U", False: ""})
    + final_profile["http_days"].gt(0).map({True: "H", False: ""})
)

final_profile["observed_channel_set"] = (
    final_profile["observed_channel_set"].replace("", "NONE")
)

final_profile.loc[
    ~has_future, "observed_channel_set"
] = "NO_ELIGIBLE_FUTURE_DAYS"

# ------------------------------------------------------------
# 5. MONTHS WITH ANY TARGET-CHANNEL ACTIVITY
# ------------------------------------------------------------

monthly_has_activity = (
    monthly[["email_days", "usb_days", "http_days"]]
    .sum(axis=1)
    .gt(0)
)

active_months = (
    monthly_has_activity
    .groupby(monthly["user"])
    .sum()
)

final_profile["active_months_any_channel"] = (
    final_profile["user"]
    .map(active_months)
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 6. CLARIFY CALENDAR-WEEKDAY STREAK NAMES
# ------------------------------------------------------------

streak_renames = {
    "max_workday_streak": "max_eligible_calendar_weekday_streak",
    "max_email_streak": "max_email_calendar_weekday_streak",
    "max_usb_streak": "max_usb_calendar_weekday_streak",
    "max_http_streak": "max_http_calendar_weekday_streak"
}

assert set(streak_renames).issubset(final_profile.columns)

final_profile = final_profile.rename(columns=streak_renames)

# ------------------------------------------------------------
# 7. COMPLETENESS CHECKS
# ------------------------------------------------------------

required_existing = [
    "user",
    "baseline_workdays",
    "future_workdays",
    "has_valid_baseline",
    "email_days",
    "usb_days",
    "http_days",
    "days_0ch",
    "days_1ch",
    "days_2ch",
    "days_3ch",
    "channel_overlap_pattern",
    "email_gap_pairs",
    "usb_gap_pairs",
    "http_gap_pairs",
    "adjacent_direct_switches",
    "active_day_direct_switches",
    "email_active_months",
    "usb_active_months",
    "http_active_months"
]

assert set(required_existing).issubset(final_profile.columns)
assert len(final_profile) == 3995
assert final_profile["user"].is_unique
assert len(activity) == 387162
assert has_future.sum() == 3737

assert (
    final_profile["active_months_any_channel"]
    .between(0, 6)
).all()

assert not final_profile.columns.duplicated().any()

# ------------------------------------------------------------
# 8. REVIEW SUMMARY — NOTHING SAVED YET
# ------------------------------------------------------------

print("=" * 72)
print("FINAL USER PROFILE — PRE-SAVE REVIEW")
print("=" * 72)

print(f"Original columns  : {len(original_columns)}")
print(f"Final columns     : {len(final_profile.columns)}")
print(f"Users retained    : {len(final_profile):,}")
print(f"Future users      : {has_future.sum():,}")
print(f"No future days    : {(~has_future).sum():,}")

print("\nEXACT DAILY CHANNEL COMBINATIONS")
print("-" * 72)

print(
    final_profile[
        [f"days_{s}" for s in channel_states]
    ].sum().to_string()
)

print("\nHISTORICAL CHANNEL BASELINE AVAILABILITY")
print("-" * 72)

for channel in ["email", "usb", "http"]:
    col = f"has_{channel}_channel_baseline"
    print(f"{channel.upper():<8}: {final_profile[col].sum():,} users")

print("\nOBSERVED CHANNEL SETS")
print("-" * 72)

print(
    final_profile["observed_channel_set"]
    .value_counts()
    .to_string()
)

print("\nVALIDATION: PASS")
print("No files saved yet.")


FINAL USER PROFILE — PRE-SAVE REVIEW
Original columns  : 87
Final columns     : 101
Users retained    : 3,995
Future users      : 3,737
No future days    : 258

EXACT DAILY CHANNEL COMBINATIONS
------------------------------------------------------------------------
days_E      198212
days_U       11822
days_H        5061
days_EU      15788
days_EH       6429
days_UH        998
days_EUH      1410

HISTORICAL CHANNEL BASELINE AVAILABILITY
------------------------------------------------------------------------
EMAIL   : 3,994 users
USB     : 783 users
HTTP    : 615 users

OBSERVED CHANNEL SETS
------------------------------------------------------------------------
observed_channel_set
E                          2742
EU                          428
EUH                         291
EH                          276
NO_ELIGIBLE_FUTURE_DAYS     258

VALIDATION: PASS
No files saved yet.



### 5.2 Final Profile Validation and Export

**Objective:** Validate and save the consolidated user behavioral profile.

**Final Checks:**
- Confirm user and Future-day coverage.
- Confirm all daily channel combinations are consistent.
- Verify that required behavioral and temporal features exist.
- Ensure users without Future workdays are explicitly identified.
- Preserve missing values for undefined statistics.
- Export the final user-level dataset and a concise audit report.

**Outputs:**
- `user_population_profile.parquet`
- `population_audit_summary.md`

**Scope:** These outputs contain only descriptive behavioral features and read-only historical baseline availability indicators. They do not determine attack eligibility or assign scenarios.


In [20]:

# ============================================================
# 5.2 FINAL PROFILE VALIDATION AND EXPORT
# ============================================================

from pathlib import Path

assert "final_profile" in globals()
assert final_profile["user"].is_unique
assert len(final_profile) == 3995
assert final_profile["future_workdays"].sum() == 387162
assert final_profile["has_valid_baseline"].all()
assert not final_profile.columns.duplicated().any()

# All seven exact channel combinations must be present.
assert all(
    f"days_{s}" in final_profile.columns
    for s in ["E", "U", "H", "EU", "EH", "UH", "EUH"]
)

# Historical channel baseline flags must be boolean.
for channel in ["email", "usb", "http"]:
    assert pd.api.types.is_bool_dtype(
        final_profile[f"has_{channel}_channel_baseline"]
    )

# Ensure zero-future users are clearly classified.
no_future = final_profile["future_workdays"].eq(0)

assert (
    final_profile.loc[
        no_future, "observed_channel_set"
    ] == "NO_ELIGIBLE_FUTURE_DAYS"
).all()

REPORT_DIR = Path(REPORT_DIR)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

profile_path = REPORT_DIR / "user_population_profile.parquet"
summary_path = REPORT_DIR / "population_audit_summary.md"

# Consolidated profile
final_profile.to_parquet(profile_path, index=False)

# Summary report
channel_sets = (
    final_profile["observed_channel_set"]
    .value_counts()
    .to_string()
)

overlap_patterns = (
    final_profile.loc[
        ~no_future, "channel_overlap_pattern"
    ].value_counts().to_string()
)

baseline_flags = "\n".join(
    f"- {c.upper()}: "
    f"{final_profile[f'has_{c}_channel_baseline'].sum():,} users"
    for c in ["email", "usb", "http"]
)

report = f"""# MUBSIR — User Population Audit

## Population

- Baseline users: {len(final_profile):,}
- Users with eligible Future workdays: {(~no_future).sum():,}
- Users without eligible Future workdays: {no_future.sum():,}
- Eligible Future user-days: {final_profile['future_workdays'].sum():,}
- Consolidated profile columns: {len(final_profile.columns)}

## Observed Channel Sets

{channel_sets}

## Channel Overlap Patterns

{overlap_patterns}

## Historical Channel Baseline Availability

{baseline_flags}

Flags require historical channel activity and valid precomputed
P50, P95, and MAD statistics. A MAD value of zero is allowed.

## Included Behavioral Features

- User and historical baseline reference
- Future workday coverage
- Channel activity days and rates
- Exact daily channel combinations
- Same-day channel overlaps
- Activity intensities and Email bytes
- Calendar-weekday streaks
- Eligible-workday and calendar-workday activity gaps
- Natural channel transitions
- Monthly temporal characteristics
- Historical per-channel baseline availability

## Interpretation Notes

- Days with zero target-channel activity may still have other activity.
- A missing gap or intensity statistic is not equivalent to zero.
- Calendar-weekday streaks and eligible-workday gaps use different
  definitions of continuity.
- Channel availability within a month does not prove that a specific
  attack sequence is feasible within a shorter window.
- June 2011 is a partial month in the dataset.

## Simulation Design Principles

- User-first, data-driven scenario selection.
- Derive feasible patterns from each user's available dates and channels.
- Do not impose predefined switching sequences.
- Preserve the original historical Baseline.
- Restrict behavioral analysis and injection to the Future period.
- Off-days mean days without attack injection.
- Final feasibility must be tested using daily data.

## Status

Population behavioral audit complete.
No attack scenario or user eligibility decision has been imposed.
"""

summary_path.write_text(report, encoding="utf-8")

# Verify saved dataset
saved_profile = pd.read_parquet(profile_path)

assert saved_profile.shape == final_profile.shape
assert saved_profile["user"].equals(final_profile["user"])
assert saved_profile.columns.equals(final_profile.columns)

print("=" * 72)
print("FINAL EXPORT — VALIDATION PASS")
print("=" * 72)
print(f"Users   : {len(saved_profile):,}")
print(f"Columns : {len(saved_profile.columns)}")
print(f"Path    : {REPORT_DIR}")
print("\nSAVED FILES:")
print(profile_path.name)
print(summary_path.name)


FINAL EXPORT — VALIDATION PASS
Users   : 3,995
Columns : 101
Path    : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\01_User_Population_Audit

SAVED FILES:
user_population_profile.parquet
population_audit_summary.md



## 6. User Population Profile Data Dictionary

**Objective:** Document the meaning, calculation, source, unit, expected values, and missing-value interpretation of every column in `user_population_profile.parquet`.

**Dictionary categories:**
- User Identity and Baseline
- Channel Activity and Frequency
- Channel Combinations and Overlap
- Activity Intensity
- Temporal Continuity
- Activity Gaps
- Natural Channel Transitions
- Monthly Activity Distribution
- Historical Channel Baseline Availability
- Future Coverage and Classification

**Documentation rules:**
- Preserve the exact column names and order from the saved Parquet file.
- Record the actual data type, observed range, example value, and missing-value count.
- Distinguish historical baseline properties from Future behavioral properties.
- Do not confuse natural channel activity with attack injections.
- Flag columns requiring manual verification rather than assigning unsupported definitions.

**Output:**
`reports/01_User_Population_Audit/user_population_profile_dictionary.csv`


In [21]:

# ============================================================
# 6.1 DEFINE USER PROFILE DATA DICTIONARY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

profile = pd.read_parquet(
    REPORT_DIR / "user_population_profile.parquet"
)

definitions = {}

FUTURE = "Future eligible user-workdays (2011-01-01 to 2011-06-01)"
BASELINE = "Historical User-Day / precomputed Baseline (read-only)"
MONTHLY = "Future monthly channel activity"
TRANSITIONS = "Future daily channel states"

def add(name, category, description, calculation,
        unit="count", valid_values="Non-negative",
        source=FUTURE, missing_meaning="Not applicable / not observed"):

    assert name not in definitions, f"Duplicate definition: {name}"

    definitions[name] = {
        "category": category,
        "description": description,
        "calculation": calculation,
        "source": source,
        "unit": unit,
        "valid_values": valid_values,
        "missing_meaning": missing_meaning,
        "review_status": "DOCUMENTED"
    }

# ------------------------------------------------------------
# A. USER IDENTITY AND BASELINE REFERENCE
# ------------------------------------------------------------

add(
    "user", "User Identity and Baseline",
    "Unique identifier of a benign CERT r6.2 user.",
    "Original user identifier; no transformation.",
    "identifier", "Unique user ID",
    BASELINE, "Not allowed"
)

add(
    "baseline_workdays", "User Identity and Baseline",
    "Number of eligible workdays in the historical Baseline.",
    "Count historical user-days where is_weekend=0 and logon_count>0.",
    "days", "Integer >= 0", BASELINE
)

add(
    "future_workdays", "Future Coverage and Classification",
    "Number of eligible workdays in the Future period.",
    "Count Future user-days where is_weekend=0 and logon_count>0.",
    "days", "Integer >= 0"
)

add(
    "has_valid_baseline", "User Identity and Baseline",
    "Whether the user has a valid general historical baseline.",
    "Previously established baseline-validity flag; not a channel-specific baseline flag.",
    "boolean", "True / False", BASELINE
)

add(
    "has_future_workdays", "Future Coverage and Classification",
    "Whether the user has at least one eligible Future workday.",
    "future_workdays > 0",
    "boolean", "True / False"
)

# ------------------------------------------------------------
# B. CHANNEL ACTIVITY AND FREQUENCY
# ------------------------------------------------------------

channel_info = {
    "email": ("External Email", "external_email_count"),
    "usb": ("USB Copy", "usb_copy_count"),
    "http": ("HTTP Upload", "www_upload_count")
}

for c, (label, count_field) in channel_info.items():

    add(
        f"{c}_days", "Channel Activity and Frequency",
        f"Number of eligible Future days with {label} activity.",
        f"Count days where {count_field} > 0.",
        "days", "Integer >= 0"
    )

    add(
        f"{c}_activity_rate", "Channel Activity and Frequency",
        f"Share of eligible Future workdays with {label} activity.",
        f"{c}_days / future_workdays. Actual scale verified during export.",
        "ratio", "0–1 if stored as a fraction",
        missing_meaning="Undefined when future_workdays=0"
    )

add(
    "active_channel_count", "Channel Activity and Frequency",
    "Number of distinct target channels observed for the user.",
    "Count channels among Email, USB, HTTP with channel_days > 0.",
    "channels", "Integer 0–3"
)

add(
    "any_channel_days", "Channel Activity and Frequency",
    "Eligible Future days with at least one active target channel.",
    "days_1ch + days_2ch + days_3ch",
    "days", "Integer >= 0"
)

add(
    "multi_channel_days", "Channel Combinations and Overlap",
    "Eligible Future days with at least two active target channels.",
    "days_2ch + days_3ch",
    "days", "Integer >= 0"
)

add(
    "three_channel_days", "Channel Combinations and Overlap",
    "Eligible Future days with all three target channels active.",
    "days_3ch",
    "days", "Integer >= 0"
)

# ------------------------------------------------------------
# C. EXACT CHANNEL COMBINATIONS
# ------------------------------------------------------------

state_info = {
    "E": "Email only",
    "U": "USB only",
    "H": "HTTP only",
    "EU": "Email and USB, but not HTTP",
    "EH": "Email and HTTP, but not USB",
    "UH": "USB and HTTP, but not Email",
    "EUH": "Email, USB and HTTP together"
}

for state, meaning in state_info.items():
    add(
        f"days_{state}", "Channel Combinations and Overlap",
        f"Number of eligible Future days with {meaning}.",
        f"Count exact daily channel_state == '{state}'.",
        "days", "Integer >= 0"
    )

for n, meaning in {
    0: "no target-channel activity",
    1: "exactly one active target channel",
    2: "exactly two active target channels",
    3: "all three target channels active"
}.items():

    add(
        f"days_{n}ch", "Channel Combinations and Overlap",
        f"Number of eligible Future days with {meaning}.",
        f"Count daily records where number of active target channels = {n}.",
        "days", "Integer >= 0",
        missing_meaning="Zero means no days in this category. days_0ch is NOT an attack Off-day."
    )

add(
    "channel_overlap_pattern", "Channel Combinations and Overlap",
    "Descriptive classification of natural channel use and same-day overlap.",
    "Classify using three-channel days, two-channel days, and distinct channels observed.",
    "category",
    "Single channel / Multiple channels on separate days only / Same-day two-channel overlap / Same-day three-channel overlap",
    missing_meaning="May be undefined for users without eligible Future workdays."
)

# ------------------------------------------------------------
# D. ACTIVITY INTENSITY
# ------------------------------------------------------------

intensity_methods = {
    "total": ("Total operations", "Sum of daily operation counts"),
    "mean": ("Mean daily operations", "Mean operation count across channel-active days"),
    "median": ("Median daily operations", "Median operation count across channel-active days"),
    "p95": ("95th percentile daily operations", "95th percentile across channel-active days"),
    "maximum": ("Maximum daily operations", "Maximum operation count across channel-active days")
}

for c, (label, count_field) in channel_info.items():

    for stat, (meaning, formula) in intensity_methods.items():

        add(
            f"{c}_{stat}_intensity", "Activity Intensity",
            f"{meaning} for {label}.",
            f"{formula} using {count_field} in the Future period.",
            "operations",
            "Non-negative numeric",
            missing_meaning="Undefined statistics may be missing when the channel has no active days."
        )

for stat, explanation in {
    "total": "Sum of recorded Email bytes over eligible Future days.",
    "mean": "Mean daily Email bytes calculated in the Email intensity audit.",
    "median": "Median daily Email bytes calculated in the Email intensity audit."
}.items():

    add(
        f"email_bytes_{stat}", "Activity Intensity",
        f"{stat.title()} recorded external Email bytes for the user.",
        explanation,
        "bytes", "Non-negative numeric",
        missing_meaning="May be missing when no eligible Email observations exist."
    )

# ------------------------------------------------------------
# E. TEMPORAL CONTINUITY
# ------------------------------------------------------------

add(
    "max_eligible_calendar_weekday_streak", "Temporal Continuity",
    "Longest run of eligible workdays consecutive in the Monday–Friday calendar.",
    "Longest sequence without missing calendar weekdays among eligible Future dates.",
    "weekdays", "Integer >= 0"
)

for c, (label, _) in channel_info.items():

    add(
        f"max_{c}_calendar_weekday_streak", "Temporal Continuity",
        f"Longest sequence of consecutive calendar weekdays with {label} activity.",
        "Longest consecutive Monday–Friday sequence where the channel is active.",
        "weekdays", "Integer >= 0"
    )

# ------------------------------------------------------------
# F. ACTIVITY GAPS
# ------------------------------------------------------------

gap_info = {
    "gap_pairs": (
        "Number of pairs of successive channel-active days.",
        "max(channel_active_days - 1, 0)",
        "pairs"
    ),
    "gap_median": (
        "Median number of eligible workdays between channel-active days.",
        "Median(current eligible position - previous active position - 1).",
        "eligible workdays"
    ),
    "gap_mean": (
        "Mean number of eligible workdays between channel-active days.",
        "Mean(current eligible position - previous active position - 1).",
        "eligible workdays"
    ),
    "gap_p90": (
        "90th percentile of eligible-workday gaps.",
        "P90 of per-pair eligible-workday gaps.",
        "eligible workdays"
    ),
    "gap_max": (
        "Longest gap measured in eligible workdays.",
        "Maximum per-pair eligible-workday gap.",
        "eligible workdays"
    ),
    "calendar_gap_median": (
        "Median gap measured in calendar weekdays.",
        "Median(busday_count(previous active date, current active date) - 1).",
        "calendar weekdays"
    ),
    "calendar_gap_max": (
        "Longest gap measured in calendar weekdays.",
        "Maximum(busday_count(previous active date, current active date) - 1).",
        "calendar weekdays"
    )
}

for c, (label, _) in channel_info.items():

    for suffix, (meaning, formula, unit) in gap_info.items():

        add(
            f"{c}_{suffix}", "Activity Gaps",
            f"{label}: {meaning}",
            formula.replace("channel_active_days", f"{c}_days"),
            unit, "Non-negative numeric",
            missing_meaning=(
                "No measurable gap when the user has fewer than two active days; "
                "gap_pairs is stored as 0, other gap statistics may be NaN."
            )
        )

# ------------------------------------------------------------
# G. NATURAL CHANNEL TRANSITIONS
# ------------------------------------------------------------

switch_definitions = {
    "adjacent_direct_switches": (
        "Direct switches between adjacent eligible workdays.",
        "Count adjacent eligible-day pairs with two different single-channel states."
    ),
    "adjacent_set_changes": (
        "Changes in active channel combinations on adjacent eligible workdays.",
        "Count adjacent eligible-day pairs with nonempty, different channel states."
    ),
    "active_day_direct_switches": (
        "Direct switches between successive target-channel-active days.",
        "Count successive active-day pairs with two different single-channel states; inactive days may lie between."
    ),
    "active_day_overlap_changes": (
        "Changed channel combinations involving multi-channel states.",
        "Count successive active-day set changes that are not direct single-channel switches."
    )
}

for name, (meaning, formula) in switch_definitions.items():

    add(
        name, "Natural Channel Transitions",
        meaning, formula,
        "transitions", "Integer >= 0",
        TRANSITIONS
    )

# ------------------------------------------------------------
# H. MONTHLY ACTIVITY DISTRIBUTION
# ------------------------------------------------------------

monthly_definitions = {
    "active_months": (
        "Number of calendar months containing channel activity.",
        "Count months with channel_days > 0.",
        "months"
    ),
    "first_active_day": (
        "First observed channel-active Future date.",
        "Minimum Future date where the channel is active.",
        "date"
    ),
    "last_active_day": (
        "Last observed channel-active Future date.",
        "Maximum Future date where the channel is active.",
        "date"
    ),
    "peak_month": (
        "Month containing the largest number of channel-active days.",
        "Month with maximum monthly channel_days; earliest month wins ties.",
        "YYYY-MM"
    ),
    "peak_month_days": (
        "Number of channel-active days in the peak month.",
        "Maximum monthly channel_days.",
        "days"
    ),
    "peak_month_share_pct": (
        "Percentage of channel-active days occurring in the peak month.",
        "100 × peak_month_days / total channel_active_days.",
        "percent"
    )
}

for c, (label, _) in channel_info.items():

    for suffix, (meaning, formula, unit) in monthly_definitions.items():

        add(
            f"{c}_{suffix}", "Monthly Activity Distribution",
            f"{label}: {meaning}",
            formula.replace("channel_days", f"{c}_days")
                   .replace("channel_active_days", f"{c}_days"),
            unit,
            "Date/month or non-negative numeric; percentage 0–100",
            MONTHLY,
            "Undefined when the channel has no active days; active_months remains 0."
        )

# ------------------------------------------------------------
# I. HISTORICAL CHANNEL BASELINE AVAILABILITY
# ------------------------------------------------------------

baseline_fields = {
    "email": ("email_active_days", "email_count_p50", "email_count_p95", "email_count_mad"),
    "usb": ("usb_active_days", "usb_copy_p50", "usb_copy_p95", "usb_copy_mad"),
    "http": ("http_upload_active_days", "http_upload_p50", "http_upload_p95", "http_upload_mad")
}

for c, fields in baseline_fields.items():

    days, p50, p95, mad = fields

    add(
        f"has_{c}_channel_baseline",
        "Historical Channel Baseline Availability",
        f"Whether historical {c.upper()} channel metrics satisfy the baseline-availability checks.",
        (
            f"({days} > 0) AND ({p50} > 0) AND ({p95} > 0) "
            f"AND ({p95} >= {p50}) AND ({mad} is not null). "
            "Zero MAD is allowed."
        ),
        "boolean", "True / False",
        "Existing user_baseline_r62_v1.parquet (read-only)",
        "False is not an automatic reason to exclude the user from attack simulation."
    )

# ------------------------------------------------------------
# J. FUTURE COVERAGE AND FINAL CLASSIFICATION
# ------------------------------------------------------------

for suffix, meaning, method in [
    (
        "first",
        "First eligible Future workday.",
        "Minimum date among the user's eligible Future workdays."
    ),
    (
        "last",
        "Last eligible Future workday.",
        "Maximum date among the user's eligible Future workdays."
    )
]:

    add(
        f"future_{suffix}_eligible_day",
        "Future Coverage and Classification",
        meaning, method,
        "date", "Future date or NaT",
        missing_meaning="NaT when the user has zero eligible Future workdays."
    )

add(
    "observed_channel_set", "Future Coverage and Classification",
    "Set of channels used by the user anywhere in the Future period.",
    (
        "Concatenate E if email_days>0, U if usb_days>0, "
        "H if http_days>0; NO_ELIGIBLE_FUTURE_DAYS if future_workdays=0."
    ),
    "category",
    "E / U / H / EU / EH / UH / EUH / NONE / NO_ELIGIBLE_FUTURE_DAYS"
)

add(
    "active_months_any_channel", "Monthly Activity Distribution",
    "Number of months with activity in at least one target channel.",
    "Count months where email_days + usb_days + http_days > 0.",
    "months", "Integer 0–6", MONTHLY
)

print("Definitions prepared:", len(definitions))
print("Profile columns     :", len(profile.columns))


Definitions prepared: 99
Profile columns     : 101



### 6.2 Generate, Inspect, and Save the Data Dictionary

**Objective:** Match definitions to the exact Parquet schema and export a column-by-column dictionary.

**Additional metadata collected from the saved dataset:**
- Actual data type
- Observed minimum and maximum
- Number of unique values
- Number of missing values
- Example value
- Definition review status

**Quality Control:**
- Exactly one dictionary entry per profile column.
- Preserve original column order.
- Do not silently omit undocumented columns.
- Mark undocumented or inconsistent definitions as `NEEDS_REVIEW`.
- Check the stored scale of activity-rate columns.
- Save the dictionary as UTF-8 CSV for compatibility with Excel and Python.


In [22]:

# ============================================================
# 6.2 EXPORT DATA DICTIONARY
# ============================================================

rows = []

for name in profile.columns:

    series = profile[name]
    nonnull = series.dropna()

    if name in definitions:
        info = definitions[name].copy()
    else:
        info = {
            "category": "NEEDS CLASSIFICATION",
            "description": "Definition requires verification.",
            "calculation": "Not confirmed from the available notebook metadata.",
            "source": "Not confirmed",
            "unit": "Not confirmed",
            "valid_values": "Inspect source calculation",
            "missing_meaning": "Not confirmed",
            "review_status": "NEEDS_REVIEW"
        }

    # Verify whether activity rates are stored as ratios or percentages.
    if name.endswith("_activity_rate") and name in definitions:

        channel = name.removesuffix("_activity_rate")

        if f"{channel}_days" in profile.columns:

            expected = (
                profile[f"{channel}_days"]
                / profile["future_workdays"].replace(0, np.nan)
            )

            mask = series.notna() & expected.notna()

            if mask.any() and np.allclose(
                series.loc[mask], expected.loc[mask],
                rtol=1e-4, atol=1e-4
            ):
                info["unit"] = "ratio"
                info["valid_values"] = "0–1"
                info["calculation"] = (
                    f"{channel}_days / future_workdays"
                )

            elif mask.any() and np.allclose(
                series.loc[mask], expected.loc[mask] * 100,
                rtol=1e-4, atol=1e-4
            ):
                info["unit"] = "percent"
                info["valid_values"] = "0–100"
                info["calculation"] = (
                    f"100 × {channel}_days / future_workdays"
                )

            else:
                info["review_status"] = "NEEDS_REVIEW"
                info["calculation"] += (
                    " [Stored values do not match the expected ratio.]"
                )

    # Observed value range
    observed_min = ""
    observed_max = ""

    if len(nonnull) and (
        pd.api.types.is_numeric_dtype(series)
        or pd.api.types.is_datetime64_any_dtype(series)
    ):
        observed_min = str(nonnull.min())
        observed_max = str(nonnull.max())

    # Short sample of actual categorical values
    observed_categories = ""

    if (
        len(nonnull)
        and (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
            or pd.api.types.is_bool_dtype(series)
        )
        and nonnull.nunique() <= 12
    ):
        observed_categories = ", ".join(
            sorted(map(str, nonnull.unique()))
        )

    rows.append({
        "column_name": name,
        "category": info["category"],
        "description": info["description"],
        "calculation": info["calculation"],
        "source": info["source"],
        "unit": info["unit"],
        "valid_values": info["valid_values"],
        "missing_meaning": info["missing_meaning"],
        "data_type": str(series.dtype),
        "observed_min": observed_min,
        "observed_max": observed_max,
        "observed_categories": observed_categories,
        "unique_values": int(series.nunique(dropna=True)),
        "null_count": int(series.isna().sum()),
        "example_value": (
            str(nonnull.iloc[0]) if len(nonnull) else ""
        ),
        "review_status": info["review_status"]
    })

data_dictionary = pd.DataFrame(rows)

# Validation
assert len(data_dictionary) == len(profile.columns) == 101
assert data_dictionary["column_name"].is_unique
assert data_dictionary["column_name"].tolist() == list(profile.columns)

needs_review = data_dictionary.loc[
    data_dictionary["review_status"] == "NEEDS_REVIEW",
    ["column_name", "category"]
]

# Save exactly one dictionary file
dictionary_path = (
    REPORT_DIR / "user_population_profile_dictionary.csv"
)

data_dictionary.to_csv(
    dictionary_path,
    index=False,
    encoding="utf-8-sig"
)

# Verify round-trip
saved_dictionary = pd.read_csv(
    dictionary_path, encoding="utf-8-sig"
)

assert len(saved_dictionary) == 101
assert (
    saved_dictionary["column_name"].tolist()
    == profile.columns.tolist()
)

print("=" * 72)
print("USER POPULATION PROFILE — DATA DICTIONARY")
print("=" * 72)

print(f"Profile columns    : {len(profile.columns)}")
print(f"Dictionary entries : {len(saved_dictionary)}")
print(
    "Documented entries:",
    (data_dictionary["review_status"] == "DOCUMENTED").sum()
)
print(f"Require review     : {len(needs_review)}")

print("\nFEATURE CATEGORIES")
print("-" * 72)
print(
    data_dictionary["category"]
    .value_counts()
    .to_string()
)

if len(needs_review):
    print("\nCOLUMNS REQUIRING REVIEW")
    print("-" * 72)
    print(needs_review.to_string(index=False))
    print("\nSTATUS: SAVED — REVIEW REQUIRED")
else:
    print("\nSTATUS: DICTIONARY COMPLETE — VALIDATION PASS")

print("\nSAVED FILE:")
print(dictionary_path.name)


USER POPULATION PROFILE — DATA DICTIONARY
Profile columns    : 101
Dictionary entries : 101
Documented entries: 99
Require review     : 2

FEATURE CATEGORIES
------------------------------------------------------------------------
category
Activity Gaps                               21
Monthly Activity Distribution               19
Activity Intensity                          18
Channel Combinations and Overlap            14
Channel Activity and Frequency               8
Future Coverage and Classification           5
Temporal Continuity                          4
Natural Channel Transitions                  4
User Identity and Baseline                   3
Historical Channel Baseline Availability     3
NEEDS CLASSIFICATION                         2

COLUMNS REQUIRING REVIEW
------------------------------------------------------------------------
          column_name             category
baseline_period_start NEEDS CLASSIFICATION
  baseline_period_end NEEDS CLASSIFICATION

STATUS: SAVED 


### 6.3 Complete Historical Baseline Period Definitions

**Objective:** Document and validate the two remaining Baseline period columns.

**Checks:**
- Inspect start and end dates.
- Confirm chronological consistency.
- Determine whether period boundaries are global or user-specific.
- Update the dictionary definitions.
- Mark both columns as documented only when their interpretation is verified.

**Output:** Updated `user_population_profile_dictionary.csv`.

The historical Baseline is read-only and will not be recalculated or modified.


In [23]:

# ============================================================
# 6.3 COMPLETE BASELINE PERIOD DEFINITIONS
# ============================================================

columns = ["baseline_period_start", "baseline_period_end"]

start = pd.to_datetime(profile[columns[0]], errors="coerce")
end = pd.to_datetime(profile[columns[1]], errors="coerce")

print("BASELINE PERIOD VERIFICATION")
print("-" * 60)

for col, dates in zip(columns, [start, end]):
    print(f"{col}:")
    print("  Unique dates :", dates.nunique())
    print("  Missing      :", dates.isna().sum())
    print("  Earliest     :", dates.min())
    print("  Latest       :", dates.max())

valid_dates = (
    start.notna().all()
    and end.notna().all()
    and (start <= end).all()
    and (end < pd.Timestamp("2011-01-01")).all()
)

global_period = (
    start.nunique() == 1
    and end.nunique() == 1
)

assert valid_dates, "Invalid historical baseline dates"

# Update the two missing dictionary entries.
for col, meaning in {
    "baseline_period_start": "Starting date",
    "baseline_period_end": "Ending date"
}.items():

    mask = data_dictionary["column_name"].eq(col)
    assert mask.sum() == 1

    updates = {
        "category": "User Identity and Baseline",
        "description": (
            f"{meaning} of the historical Baseline period "
            "recorded for the user."
        ),
        "calculation": (
            "Historical Baseline period boundary carried forward "
            "from the existing population profile; not recalculated."
        ),
        "source": "Historical Baseline metadata (read-only)",
        "unit": "date",
        "valid_values": "Historical date before 2011-01-01",
        "missing_meaning": "Missing historical period boundary.",
        "review_status": (
            "DOCUMENTED" if global_period else "NEEDS_REVIEW"
        )
    }

    for field, value in updates.items():
        data_dictionary.loc[mask, field] = value

# Export updated dictionary
data_dictionary.to_csv(
    dictionary_path,
    index=False,
    encoding="utf-8-sig"
)

remaining = (
    data_dictionary["review_status"] == "NEEDS_REVIEW"
).sum()

print("\nFINAL DICTIONARY REVIEW")
print("-" * 60)
print("Total columns      :", len(data_dictionary))
print("Documented columns :", len(data_dictionary) - remaining)
print("Require review     :", remaining)
print("Global period      :", global_period)
print("Saved              :", dictionary_path.name)

if remaining == 0:
    print("\nVALIDATION: PASS — DICTIONARY COMPLETE")
else:
    print("\nREVIEW REQUIRED — check user-specific period definitions")


BASELINE PERIOD VERIFICATION
------------------------------------------------------------
baseline_period_start:
  Unique dates : 1
  Missing      : 0
  Earliest     : 2010-01-02 00:00:00
  Latest       : 2010-01-02 00:00:00
baseline_period_end:
  Unique dates : 1
  Missing      : 0
  Earliest     : 2010-12-31 00:00:00
  Latest       : 2010-12-31 00:00:00

FINAL DICTIONARY REVIEW
------------------------------------------------------------
Total columns      : 101
Documented columns : 101
Require review     : 0
Global period      : True
Saved              : user_population_profile_dictionary.csv

VALIDATION: PASS — DICTIONARY COMPLETE
